# Data processing

In [30]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import torch
torch.use_deterministic_algorithms(False)

In [1]:
import warnings
warnings.filterwarnings('ignore')

import torch
print(f'CUDA Availale: {torch.cuda.is_available()}')

import pandas as pd 

sentiment_indicator_stock = pd.read_csv('sentiment_indicator_stock.csv', parse_dates=['Date'])
sentiment_indicator_stock.columns

CUDA Availale: True


Index(['Date', 'Symbol', 'Close', 'Volume', 'MACD', 'RSI', 'CCI', 'ADX',
       'Sentiment_Label'],
      dtype='object')

In [2]:
# feature importance using XGboost

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

class TFTStockSelector:
    def __init__(self, lookback_days=60, forecast_days=21, min_train_samples=1000, 
                 alpha=0.4, validation_splits=5):
        """
        Stock selector optimized for TFT time series forecasting
        
        Parameters:
        - lookback_days: TFT lookback window (60 days)
        - forecast_days: TFT forecast horizon (21 days) 
        - min_train_samples: minimum samples per ticker for inclusion
        - alpha: weight between predictability and stability in composite score
        - validation_splits: number of time series validation folds
        """
        self.lookback_days = lookback_days
        self.forecast_days = forecast_days
        self.min_train_samples = min_train_samples
        self.alpha = alpha
        self.validation_splits = validation_splits

    def prepare_data(self, df):
        """Prepare data with time features for TFT-style analysis"""
        df = df.copy()
        
        # Ensure datetime and sort
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values(['Symbol', 'Date']).reset_index(drop=True)
        
        # Drop missing values in core features
        required_cols = ['MACD', 'RSI', 'CCI', 'ADX', 'Sentiment_Label', 'Close']
        df = df.dropna(subset=required_cols)
        
        # Create time features (like you'll use in TFT)
        df['time_idx'] = df.groupby('Symbol').cumcount()
        df['month'] = df['Date'].dt.month
        df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12)
        df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12)
        df['day_of_week'] = df['Date'].dt.dayofweek
        df['day_of_year'] = df['Date'].dt.dayofyear
        
        # Create target variable (closing price prediction)
        df['target'] = df.groupby('Symbol')['Close'].shift(-self.forecast_days)
        df = df.dropna(subset=['target'])
        
        # Filter tickers with insufficient data
        ticker_counts = df['Symbol'].value_counts()
        valid_tickers = ticker_counts[ticker_counts >= self.min_train_samples].index
        df = df[df['Symbol'].isin(valid_tickers)]
        
        print(f"Data prepared: {len(df):,} samples across {len(valid_tickers)} tickers")
        print(f"Date range: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")
        print(f"Average samples per ticker: {len(df) // len(valid_tickers):,}")
        
        return df

    def create_features(self, df):
        """Create feature matrix for predictability analysis"""
        # Technical features
        technical_features = ['MACD', 'RSI', 'CCI', 'ADX', 'Sentiment_Label']
        
        # Time features 
        time_features = ['time_idx', 'sin_month', 'cos_month', 'day_of_week', 'day_of_year']
        
        # Price-based features
        for symbol in df['Symbol'].unique():
            df.loc[df['Symbol'] == symbol, 'price_lag_1'] = df.loc[df['Symbol'] == symbol, 'Close'].shift(1)
            df.loc[df['Symbol'] == symbol, 'price_lag_7'] = df.loc[df['Symbol'] == symbol, 'Close'].shift(7)
            df.loc[df['Symbol'] == symbol, 'price_lag_21'] = df.loc[df['Symbol'] == symbol, 'Close'].shift(21)
            df.loc[df['Symbol'] == symbol, 'price_ma_5'] = df.loc[df['Symbol'] == symbol, 'Close'].rolling(5).mean()
            df.loc[df['Symbol'] == symbol, 'price_ma_20'] = df.loc[df['Symbol'] == symbol, 'Close'].rolling(20).mean()
            df.loc[df['Symbol'] == symbol, 'volatility_20'] = df.loc[df['Symbol'] == symbol, 'Close'].rolling(20).std()
        
        price_features = ['price_lag_1', 'price_lag_7', 'price_lag_21', 'price_ma_5', 'price_ma_20', 'volatility_20']
        
        # Stock identifier features (for individual stock predictability)
        tickers = sorted(df['Symbol'].unique())
        for ticker in tickers:
            df[f'is_{ticker}'] = (df['Symbol'] == ticker).astype(int)
        
        stock_features = [f'is_{ticker}' for ticker in tickers]
        
        all_features = technical_features + time_features + price_features + stock_features
        
        # Remove rows with NaN values created by lags/rolling
        df = df.dropna()
        
        print(f"Features created: {len(all_features)} total")
        print(f"- Technical: {len(technical_features)}")
        print(f"- Time: {len(time_features)}")  
        print(f"- Price: {len(price_features)}")
        print(f"- Stock identifiers: {len(stock_features)}")
        
        return df, all_features, tickers

    def time_series_validation(self, df, features, tickers):
        """Time series cross-validation to assess ticker predictability"""
        print("Performing time series validation...")
        
        # Initialize storage for metrics
        ticker_metrics = {ticker: {
            'mse_scores': [],
            'mae_scores': [], 
            'r2_scores': [],
            'feature_importance': [],
            'sample_counts': [],
            'prediction_stability': []
        } for ticker in tickers}
        
        # Time series split
        tscv = TimeSeriesSplit(n_splits=self.validation_splits, test_size=None)
        
        # Get unique dates for splitting
        dates = sorted(df['Date'].unique())
        
        fold = 0
        for train_idx, test_idx in tqdm(tscv.split(dates), desc="Time series folds", total=self.validation_splits):
            fold += 1
            
            train_dates = [dates[i] for i in train_idx]
            test_dates = [dates[i] for i in test_idx]
            
            train_df = df[df['Date'].isin(train_dates)]
            test_df = df[df['Date'].isin(test_dates)]
            
            # Skip if insufficient data
            if len(train_df) < 1000 or len(test_df) < 100:
                continue
                
            # Prepare features and target
            X_train = train_df[features].values
            y_train = train_df['target'].values
            X_test = test_df[features].values  
            y_test = test_df['target'].values
            
            # Scale features
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            # Train XGBoost for regression
            model = xgb.XGBRegressor(
                objective='reg:squarederror',
                n_estimators=100,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                verbosity=0
            )
            
            model.fit(X_train_scaled, y_train)
            
            # Make predictions
            y_pred = model.predict(X_test_scaled)
            
            # Get feature importance
            booster = model.get_booster()
            importance_dict = booster.get_score(importance_type='gain')
            total_importance = sum(importance_dict.values()) + 1e-12
            
            # Calculate metrics for each ticker
            for ticker in tickers:
                # Get ticker-specific test data
                ticker_mask = test_df['Symbol'] == ticker
                ticker_test = test_df[ticker_mask]
                
                if len(ticker_test) < 5:  # Skip if too few samples
                    continue
                
                # Get predictions for this ticker
                ticker_indices = ticker_test.index
                test_indices = [i for i, idx in enumerate(test_df.index) if idx in ticker_indices]
                
                if not test_indices:
                    continue
                    
                ticker_y_true = ticker_test['target'].values
                ticker_y_pred = y_pred[test_indices]
                
                # Calculate regression metrics
                try:
                    mse = mean_squared_error(ticker_y_true, ticker_y_pred)
                    mae = mean_absolute_error(ticker_y_true, ticker_y_pred)
                    r2 = r2_score(ticker_y_true, ticker_y_pred)
                    
                    # Feature importance for this ticker's identifier
                    ticker_feature = f'is_{ticker}'
                    ticker_importance = importance_dict.get(ticker_feature, 0.0) / total_importance
                    
                    # Prediction stability (lower std of residuals = more stable)
                    residuals = ticker_y_true - ticker_y_pred
                    stability = 1 / (1 + np.std(residuals))
                    
                    # Store metrics
                    ticker_metrics[ticker]['mse_scores'].append(mse)
                    ticker_metrics[ticker]['mae_scores'].append(mae)
                    ticker_metrics[ticker]['r2_scores'].append(r2)
                    ticker_metrics[ticker]['feature_importance'].append(ticker_importance)
                    ticker_metrics[ticker]['sample_counts'].append(len(ticker_test))
                    ticker_metrics[ticker]['prediction_stability'].append(stability)
                except:
                    # Skip if calculation fails
                    continue
        
        return ticker_metrics

    def compute_ticker_scores(self, ticker_metrics, df):
        """Compute comprehensive scores for ticker selection"""
        results = []
        
        for ticker, metrics in ticker_metrics.items():
            if not metrics['mse_scores']:  # Skip tickers with no data
                continue
            
            # Basic performance metrics
            avg_mse = np.mean(metrics['mse_scores'])
            avg_mae = np.mean(metrics['mae_scores'])
            avg_r2 = np.mean(metrics['r2_scores'])
            avg_importance = np.mean(metrics['feature_importance'])
            avg_samples = np.mean(metrics['sample_counts'])
            avg_stability = np.mean(metrics['prediction_stability'])
            
            # Consistency metrics (lower std = more consistent)
            r2_consistency = 1 / (1 + np.std(metrics['r2_scores']))
            mse_consistency = 1 / (1 + np.std(metrics['mse_scores']))
            
            # Data quality metrics
            ticker_data = df[df['Symbol'] == ticker]
            total_samples = len(ticker_data)
            date_range = (ticker_data['Date'].max() - ticker_data['Date'].min()).days
            data_density = total_samples / max(date_range, 1) if date_range > 0 else 0
            
            # Price characteristics
            price_volatility = ticker_data['Close'].std() / ticker_data['Close'].mean()
            price_trend = np.corrcoef(range(len(ticker_data)), ticker_data['Close'].values)[0, 1]
            
            # Composite predictability score
            predictability_score = (
                0.4 * max(0, avg_r2) +  # R² performance
                0.3 * r2_consistency +   # Consistency across folds
                0.2 * avg_stability +    # Prediction stability  
                0.1 * min(1.0, avg_importance * 100)  # Feature importance
            )
            
            # Data quality score
            quality_score = (
                0.5 * min(1.0, total_samples / 2000) +  # Sample adequacy
                0.3 * min(1.0, data_density) +          # Data density
                0.2 * min(1.0, abs(price_trend))        # Price trend strength
            )
            
            # Final composite score
            composite_score = (
                self.alpha * predictability_score +
                (1 - self.alpha) * quality_score
            )
            
            results.append({
                'Symbol': ticker,
                'Avg_R2': avg_r2,
                'Avg_MSE': avg_mse,
                'Avg_MAE': avg_mae,
                'R2_Consistency': r2_consistency,
                'MSE_Consistency': mse_consistency,
                'Prediction_Stability': avg_stability,
                'Feature_Importance': avg_importance,
                'Total_Samples': total_samples,
                'Data_Density': data_density,
                'Price_Volatility': price_volatility,
                'Price_Trend': abs(price_trend),
                'Predictability_Score': predictability_score,
                'Quality_Score': quality_score,
                'Composite_Score': composite_score
            })
        
        df_results = pd.DataFrame(results)
        df_results = df_results.sort_values('Composite_Score', ascending=False).reset_index(drop=True)
        
        return df_results

    def final_validation(self, df, features, selected_tickers):
        """Final validation on holdout period"""
        print("Performing final validation...")
        
        # Use last 20% of data as holdout
        dates = sorted(df['Date'].unique())
        holdout_split = int(len(dates) * 0.8)
        
        train_dates = dates[:holdout_split]
        holdout_dates = dates[holdout_split:]
        
        train_df = df[df['Date'].isin(train_dates)]
        holdout_df = df[df['Date'].isin(holdout_dates)]
        
        print(f"Final validation: {len(train_dates)} train days, {len(holdout_dates)} holdout days")
        
        # Train final model
        X_train = train_df[features].values
        y_train = train_df['target'].values
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        
        final_model = xgb.XGBRegressor(
            objective='reg:squarederror',
            n_estimators=150,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbosity=0
        )
        
        final_model.fit(X_train_scaled, y_train)
        
        # Validate selected tickers
        validation_results = []
        
        for ticker in selected_tickers:
            ticker_holdout = holdout_df[holdout_df['Symbol'] == ticker]
            
            if len(ticker_holdout) < 10:
                continue
                
            X_holdout = scaler.transform(ticker_holdout[features].values)
            y_holdout = ticker_holdout['target'].values
            
            y_pred = final_model.predict(X_holdout)
            
            # Calculate metrics
            mse = mean_squared_error(y_holdout, y_pred)
            mae = mean_absolute_error(y_holdout, y_pred)
            r2 = r2_score(y_holdout, y_pred)
            
            # Price-based metrics
            mape = np.mean(np.abs((y_holdout - y_pred) / y_holdout)) * 100
            rmse = np.sqrt(mse)
            
            validation_results.append({
                'Symbol': ticker,
                'Holdout_R2': r2,
                'Holdout_MSE': mse,
                'Holdout_RMSE': rmse,
                'Holdout_MAE': mae,
                'Holdout_MAPE': mape,
                'Holdout_Samples': len(ticker_holdout)
            })
        
        return pd.DataFrame(validation_results)

    def select_tickers_for_tft(self, df_raw, top_k=8, min_r2_threshold=0.1):
        """Main method to select best tickers for TFT optimization"""
        print("="*70)
        print("TICKER SELECTION FOR TFT (TEMPORAL FUSION TRANSFORMER) OPTIMIZATION")
        print("="*70)
        
        # Prepare data
        df = self.prepare_data(df_raw)
        df, features, tickers = self.create_features(df)
        
        print(f"\nAvailable tickers: {tickers}")
        print(f"Total features: {len(features)}")
        print(f"TFT Configuration: {self.lookback_days} days lookback, {self.forecast_days} days forecast")
        
        # Time series validation
        ticker_metrics = self.time_series_validation(df, features, tickers)
        
        # Compute scores
        scores_df = self.compute_ticker_scores(ticker_metrics, df)
        
        # Apply R² threshold filter
        qualified_tickers = scores_df[scores_df['Avg_R2'] >= min_r2_threshold]
        
        if len(qualified_tickers) < top_k:
            print(f"\nWarning: Only {len(qualified_tickers)} tickers meet R² threshold of {min_r2_threshold}")
            print("Selecting top tickers regardless of threshold...")
            selected_tickers_df = scores_df.head(top_k)
        else:
            selected_tickers_df = qualified_tickers.head(top_k)
        
        selected_tickers = selected_tickers_df['Symbol'].tolist()
        
        # Final validation
        validation_results = self.final_validation(df, features, selected_tickers)
        
        # Display results
        print("\n" + "="*70)
        print("SELECTED TICKERS FOR TFT OPTUNA OPTIMIZATION")
        print("="*70)
        
        display_cols = ['Symbol', 'Avg_R2', 'R2_Consistency', 'Prediction_Stability', 
                       'Total_Samples', 'Composite_Score']
        print(selected_tickers_df[display_cols].round(4).to_string(index=False))
        
        if not validation_results.empty:
            print(f"\n{'='*50}")
            print("HOLDOUT VALIDATION RESULTS")
            print("="*50)
            val_cols = ['Symbol', 'Holdout_R2', 'Holdout_RMSE', 'Holdout_MAPE', 'Holdout_Samples']
            print(validation_results[val_cols].round(4).to_string(index=False))
        
        print(f"\n{'='*50}")
        print("SUMMARY FOR TFT OPTIMIZATION")
        print("="*50)
        print(f"Selected Tickers: {selected_tickers}")
        print(f"Average R²: {selected_tickers_df['Avg_R2'].mean():.4f}")
        print(f"Average Consistency: {selected_tickers_df['R2_Consistency'].mean():.4f}")
        print(f"Average Samples per Ticker: {selected_tickers_df['Total_Samples'].mean():,.0f}")
        
        print(f"\nThese tickers are optimized for:")
        print(f"✓ TFT time series forecasting ({self.lookback_days}→{self.forecast_days} days)")
        print(f"✓ Closing price prediction accuracy")
        print(f"✓ Consistent performance across time periods")
        print(f"✓ Sufficient historical data for training")
        
        return selected_tickers, selected_tickers_df, validation_results


# =============================================================================
# MAIN EXECUTION 
# =============================================================================

if __name__ == "__main__":
    print("Loading TFT Stock Selector...")
    print("="*50)
    
    # 1. LOAD YOUR DATA
    print("Step 1: Loading data...")
    try:
        #data loading
        df = pd.read_csv('sentiment_indicator_stock.csv')
        print(f"Data loaded: {len(df)} rows, {len(df.columns)} columns")
        print(f"Columns: {list(df.columns)}")
        print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
        print(f"Unique tickers: {sorted(df['Symbol'].unique())}")
    except Exception as e:
        print(f"Error loading data: {e}")
        exit()
    
    # 2. INITIALIZE SELECTOR
    print("\nStep 2: Initializing TFT selector...")
    selector = TFTStockSelector(
        lookback_days=60,           #  TFT lookback window
        forecast_days=21,           #  TFT forecast horizon  
        min_train_samples=3000,     # Minimum samples per ticker  
        alpha=0.7,                  # Weight: 0.6=predictability, 0.4=data quality
        validation_splits=5         # Number of time series CV folds
    )
    print("✓ Selector initialized")
    
    # 3. RUN STOCK SELECTION
    print("\nStep 3: Running ticker selection for TFT...")
    print("This may take 5-15 minutes depending on data size...")
    
    try:
        selected_tickers, scores_df, validation_df = selector.select_tickers_for_tft(
            df,
            top_k=8,                    # Select 8 best tickers for Optuna
            min_r2_threshold=0.05       # Minimum R² threshold (adjust as needed)
        )
        
        print("\n" + "="*70)
        print("SELECTION COMPLETE!")
        print("="*70)
        
        # 4. SAVE RESULTS
        print("\nStep 4: Saving results...")
        scores_df.to_csv('selected_tickers_scores.csv', index=False)
        if not validation_df.empty:
            validation_df.to_csv('validation_results.csv', index=False)
        
        # Save just the ticker list for easy access
        with open('selected_tickers_for_optuna.txt', 'w') as f:
            for ticker in selected_tickers:
                f.write(f"{ticker}\n")
        
        print("Results saved:")
        print("  - selected_tickers_scores.csv")
        print("  - validation_results.csv") 
        print("  - selected_tickers_for_optuna.txt")
        
        print(f"\nFINAL RESULT: {selected_tickers}")
        print("These tickers are ready for TFT Optuna optimization!")
        
    except Exception as e:
        print(f"Error during selection: {e}")
        import traceback
        traceback.print_exc()

Loading TFT Stock Selector...
Step 1: Loading data...
Data loaded: 109100 rows, 9 columns
Columns: ['Date', 'Symbol', 'Close', 'Volume', 'MACD', 'RSI', 'CCI', 'ADX', 'Sentiment_Label']
Date range: 2003-04-01 to 2025-03-28
Unique tickers: ['ACC.NS', 'AXISBANK.NS', 'BHEL.NS', 'CIPLA.NS', 'DRREDDY.NS', 'GAIL.NS', 'GRASIM.NS', 'HDFCBANK.NS', 'HINDUNILVR.NS', 'ICICIBANK.NS', 'INFY.NS', 'KOTAKBANK.NS', 'LT.NS', 'MRF.NS', 'NCC.NS', 'PNB.NS', 'RELIANCE.NS', 'SBIN.NS', 'TVSMOTOR.NS', 'WIPRO.NS']

Step 2: Initializing TFT selector...
✓ Selector initialized

Step 3: Running ticker selection for TFT...
This may take 5-15 minutes depending on data size...
TICKER SELECTION FOR TFT (TEMPORAL FUSION TRANSFORMER) OPTIMIZATION
Data prepared: 108,680 samples across 20 tickers
Date range: 2003-04-01 to 2025-02-25
Average samples per ticker: 5,434
Features created: 36 total
- Technical: 5
- Time: 5
- Price: 6
- Stock identifiers: 20

Available tickers: ['ACC.NS', 'AXISBANK.NS', 'BHEL.NS', 'CIPLA.NS', 'DRRE

Time series folds: 100%|██████████| 5/5 [00:03<00:00,  1.39it/s]


Performing final validation...
Final validation: 4330 train days, 1083 holdout days

SELECTED TICKERS FOR TFT OPTUNA OPTIMIZATION
      Symbol  Avg_R2  R2_Consistency  Prediction_Stability  Total_Samples  Composite_Score
 TVSMOTOR.NS  0.8128          0.9123                0.0731           5413           0.6857
 HDFCBANK.NS  0.7352          0.9046                0.0319           5413           0.6681
     INFY.NS  0.7208          0.8856                0.0298           5413           0.6554
 RELIANCE.NS  0.6861          0.8687                0.0369           5413           0.6427
      NCC.NS  0.7426          0.8669                0.0817           5413           0.6415
  DRREDDY.NS  0.7065          0.7664                0.0265           5413           0.6292
     SBIN.NS  0.6542          0.8465                0.0386           5413           0.6285
ICICIBANK.NS  0.6404          0.8233                0.0405           5413           0.6200

HOLDOUT VALIDATION RESULTS
      Symbol  Holdout_R

In [3]:
# debug data

import pandas as pd
import numpy as np

def debug_data_preparation(file_path):
    print("="*60)
    print("DEBUGGING DATA PREPARATION")
    print("="*60)
    
    # 1. Load raw data
    print("1. Loading raw data...")
    try:
        df_raw = pd.read_csv(file_path)
        print(f"✓ Raw data loaded: {len(df_raw):,} rows, {len(df_raw.columns)} columns")
        print(f"✓ Columns: {list(df_raw.columns)}")
        print(f"✓ Unique symbols: {len(df_raw['Symbol'].unique())} symbols")
        print(f"✓ Symbol counts:\n{df_raw['Symbol'].value_counts().head(10)}")
    except Exception as e:
        print(f"Error loading data: {e}")
        return
    
    # 2. Check data types and missing values
    print("\n2. Data quality check...")
    print("Missing values per column:")
    missing_info = df_raw.isnull().sum()
    for col, missing in missing_info.items():
        if missing > 0:
            print(f"  {col}: {missing:,} missing ({missing/len(df_raw)*100:.1f}%)")
        else:
            print(f"  {col}: 0 missing")
    
    # 3. Date conversion and sorting
    print("\n3. Converting dates...")
    df = df_raw.copy()
    try:
        df['Date'] = pd.to_datetime(df['Date'])
        print(f"✓ Date conversion successful")
        print(f"✓ Date range: {df['Date'].min()} to {df['Date'].max()}")
        print(f"✓ Total days: {(df['Date'].max() - df['Date'].min()).days}")
    except Exception as e:
        print(f"Date conversion failed: {e}")
        print(f"Sample Date values: {df_raw['Date'].head()}")
        return
    
    df = df.sort_values(['Symbol', 'Date']).reset_index(drop=True)
    print(f"✓ Data sorted: {len(df):,} rows")
    
    # 4. Drop missing values
    print("\n4. Dropping missing values...")
    required_cols = ['MACD', 'RSI', 'CCI', 'ADX', 'Sentiment_Label', 'Close']
    print(f"Required columns: {required_cols}")
    
    before_drop = len(df)
    df = df.dropna(subset=required_cols)
    after_drop = len(df)
    dropped = before_drop - after_drop
    
    print(f"✓ Before dropping: {before_drop:,} rows")
    print(f"✓ After dropping: {after_drop:,} rows")
    print(f"✓ Dropped: {dropped:,} rows ({dropped/before_drop*100:.1f}%)")
    
    # 5. Create time features
    print("\n5. Creating time features...")
    df['time_idx'] = df.groupby('Symbol').cumcount()
    df['month'] = df['Date'].dt.month
    df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12)
    df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12)
    print(f"✓ Time features created: {len(df):,} rows")
    
    # 6. Create target (THIS IS WHERE DATA GETS LOST!)
    print("\n6. Creating target variable (21-day ahead Close price)...")
    forecast_days = 21
    before_target = len(df)
    
    df['target'] = df.groupby('Symbol')['Close'].shift(-forecast_days)
    
    # Check how many NaN targets were created
    nan_targets = df['target'].isnull().sum()
    print(f"✓ Target created with {nan_targets:,} NaN values")
    
    # Drop NaN targets
    df = df.dropna(subset=['target'])
    after_target = len(df)
    target_dropped = before_target - after_target
    
    print(f"✓ Before target drop: {before_target:,} rows")
    print(f"✓ After target drop: {after_target:,} rows")
    print(f"✓ Lost to target creation: {target_dropped:,} rows ({target_dropped/before_target*100:.1f}%)")
    
    # 7. Filter by minimum samples
    print("\n7. Filtering by minimum samples per ticker...")
    min_samples = 2000  # Your setting
    ticker_counts = df['Symbol'].value_counts()
    print(f"Samples per ticker AFTER target creation:")
    for symbol, count in ticker_counts.items():
        status = "✓ KEEP" if count >= min_samples else "❌ DROP"
        print(f"  {symbol}: {count:,} samples {status}")
    
    valid_tickers = ticker_counts[ticker_counts >= min_samples].index
    before_filter = len(df)
    df = df[df['Symbol'].isin(valid_tickers)]
    after_filter = len(df)
    filter_dropped = before_filter - after_filter
    
    print(f"\n✓ Before ticker filter: {before_filter:,} rows")
    print(f"✓ After ticker filter: {after_filter:,} rows")
    print(f"✓ Lost to ticker filter: {filter_dropped:,} rows ({filter_dropped/before_filter*100:.1f}%)")
    print(f"✓ Valid tickers: {list(valid_tickers)} ({len(valid_tickers)} tickers)")
    
    # 8. Create lag features (MORE DATA LOSS!)
    print("\n8. Creating lag features...")
    before_lags = len(df)
    
    for symbol in df['Symbol'].unique():
        mask = df['Symbol'] == symbol
        df.loc[mask, 'price_lag_1'] = df.loc[mask, 'Close'].shift(1)
        df.loc[mask, 'price_lag_7'] = df.loc[mask, 'Close'].shift(7)
        df.loc[mask, 'price_lag_21'] = df.loc[mask, 'Close'].shift(21)
        df.loc[mask, 'price_ma_5'] = df.loc[mask, 'Close'].rolling(5).mean()
        df.loc[mask, 'price_ma_20'] = df.loc[mask, 'Close'].rolling(20).mean()
        df.loc[mask, 'volatility_20'] = df.loc[mask, 'Close'].rolling(20).std()
    
    # Drop rows with NaN from lags
    df = df.dropna()
    after_lags = len(df)
    lag_dropped = before_lags - after_lags
    
    print(f"✓ Before lag features: {before_lags:,} rows")
    print(f"✓ After lag features: {after_lags:,} rows")
    print(f"✓ Lost to lag features: {lag_dropped:,} rows ({lag_dropped/before_lags*100:.1f}%)")
    
    # 9. Final summary
    print("\n" + "="*60)
    print("FINAL DATA SUMMARY")
    print("="*60)
    print(f"Original data: {len(df_raw):,} rows")
    print(f"Final data: {len(df):,} rows")
    print(f"Total data loss: {len(df_raw) - len(df):,} rows ({(len(df_raw) - len(df))/len(df_raw)*100:.1f}%)")
    
    if len(df) < 10000:
        print(" WARNING: Very small dataset! This explains fast execution.")
        print("   Possible issues:")
        print("   1. Many missing values in technical indicators")
        print("   2. Target creation drops last 21 rows per ticker")
        print("   3. Lag features drop first 21 rows per ticker")
        print("   4. min_train_samples=2000 might be too high")
    
    final_ticker_counts = df['Symbol'].value_counts()
    print(f"\nFinal samples per ticker:")
    for symbol, count in final_ticker_counts.items():
        print(f"  {symbol}: {count:,} samples")
    
    return df

# RUN THIS FIRST
if __name__ == "__main__":
    # Debug your data
    df_debug = debug_data_preparation('sentiment_indicator_stock.csv')
    
    print("\n" + "="*60)
    print("RECOMMENDATIONS")
    print("="*60)
    
    if df_debug is not None and len(df_debug) < 10000:
        print("To get more data:")
        print("1. Reduce min_train_samples from 2000 to 500:")
        print("   selector = TFTStockSelector(min_train_samples=500)")
        print("")
        print("2. Reduce forecast_days from 21 to 5:")
        print("   selector = TFTStockSelector(forecast_days=5)")
        print("")
        print("3. Check your data for missing technical indicators")
        print("4. Consider using fewer lag features")
    else: 
        print('Nothing, data is good to go')

DEBUGGING DATA PREPARATION
1. Loading raw data...
✓ Raw data loaded: 109,100 rows, 9 columns
✓ Columns: ['Date', 'Symbol', 'Close', 'Volume', 'MACD', 'RSI', 'CCI', 'ADX', 'Sentiment_Label']
✓ Unique symbols: 20 symbols
✓ Symbol counts:
RELIANCE.NS    5455
BHEL.NS        5455
TVSMOTOR.NS    5455
GRASIM.NS      5455
ACC.NS         5455
MRF.NS         5455
GAIL.NS        5455
WIPRO.NS       5455
PNB.NS         5455
DRREDDY.NS     5455
Name: Symbol, dtype: int64

2. Data quality check...
Missing values per column:
  Date: 0 missing
  Symbol: 0 missing
  Close: 0 missing
  Volume: 0 missing
  MACD: 0 missing
  RSI: 0 missing
  CCI: 0 missing
  ADX: 0 missing
  Sentiment_Label: 0 missing

3. Converting dates...
✓ Date conversion successful
✓ Date range: 2003-04-01 00:00:00 to 2025-03-28 00:00:00
✓ Total days: 8032
✓ Data sorted: 109,100 rows

4. Dropping missing values...
Required columns: ['MACD', 'RSI', 'CCI', 'ADX', 'Sentiment_Label', 'Close']
✓ Before dropping: 109,100 rows
✓ After dropp

# Final pipeline

In [ ]:
# main pipeline 

import pandas as pd
import numpy as np
import warnings
import logging
import json
from datetime import datetime
from pathlib import Path
from typing import Dict, Any, Optional, Tuple, List
import matplotlib.pyplot as plt

import optuna
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import MAE, RMSE, MAPE

# SETUP
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ENHANCED CONFIGURATION
class Config:
    CSV_FILE = 'sentiment_indicator_stock.csv'
    SELECTED_TICKERS_FILE = 'selected_tickers_for_optuna.txt'  # New: file with selected tickers
    MAX_ENCODER_LENGTH = 60
    MAX_PREDICTION_LENGTH = 21
    BATCH_SIZE = 128
    N_TRIALS = 20
    RANDOM_STATE = 42
    
    # Model checkpointing
    CHECKPOINT_DIR = Path("./checkpoints")
    CHECKPOINT_DIR.mkdir(exist_ok=True)
    
    # Early stopping patience
    EARLY_STOPPING_PATIENCE = 10
    
    # NEW: Optuna-specific settings
    USE_SELECTED_TICKERS_FOR_OPTUNA = True  # Whether to use selected tickers for optimization
    OPTUNA_TICKER_SELECTION = "selected"  # "selected", "all", or "random"
    RANDOM_TICKER_COUNT = 10  # Number of random tickers to use if OPTUNA_TICKER_SELECTION = "random"

config = Config()

# Global variables
optimization_results = {}
training_data_global = None
validation_data_global = None
df_clean_global = None
selected_tickers_global = None

print("\n ENHANCED PIPELINE - TICKER SELECTION FOR OPTUNA")
print("=" * 70)
print("Available functions:")
print("1. load_selected_tickers() - Load ticker list from file")
print("2. preview_data() - Preview and analyze your data")
print("3. preview_selected_data() - Preview data for selected tickers only")
print("4. run_optimization() - Start hyperparameter optimization (uses selected tickers)")
print("5. show_best_params() - View best parameters")
print("6. train_final_model_with_best_params() - Train final model (can use all data)")
print("7. train_final_model_with_custom_params(params) - Train with custom params")
print("8. plot_predictions(model, n_samples=5) - Visualize predictions")
print("9. evaluate_model(model) - Comprehensive model evaluation")
print("10. export_model(model, filename) - Export trained model")
print("11. quick_train_selected() - Fast pipeline with selected tickers")
print("12. set_ticker_selection_mode(mode) - Change ticker selection mode")
print("=" * 70)

#------------------------------------------------
# NEW: TICKER SELECTION FUNCTIONS
#------------------------------------------------
def load_selected_tickers(filename: str = None) -> List[str]:
    """Load selected tickers from file."""
    global selected_tickers_global
    
    if filename is None:
        filename = config.SELECTED_TICKERS_FILE
    
    try:
        with open(filename, 'r') as f:
            # Handle different file formats
            content = f.read().strip()
            
            # Try different parsing methods
            if ',' in content:
                # Comma-separated
                tickers = [ticker.strip().upper() for ticker in content.split(',')]
            elif '\n' in content:
                # Line-separated
                tickers = [ticker.strip().upper() for ticker in content.split('\n')]
            else:
                # Space-separated or single ticker
                tickers = [ticker.strip().upper() for ticker in content.split()]
            
            # Remove empty strings
            tickers = [ticker for ticker in tickers if ticker]
            
            selected_tickers_global = tickers
            
            print(f"✓ Loaded {len(tickers)} selected tickers from '{filename}':")
            print(f"   {tickers}")
            
            return tickers
            
    except FileNotFoundError:
        print(f"✗ File '{filename}' not found!")
        print("   Please create a file with your selected tickers (one per line or comma-separated)")
        print("   Example content:")
        print("   AAPL")
        print("   GOOGL")
        print("   MSFT")
        return []
    except Exception as e:
        print(f"✗ Error loading tickers: {str(e)}")
        return []

def get_optimization_tickers(df: pd.DataFrame) -> List[str]:
    """Get tickers to use for optimization based on configuration."""
    global selected_tickers_global
    
    all_tickers = sorted(df['Ticker'].unique().tolist())
    
    if config.OPTUNA_TICKER_SELECTION == "all":
        print(f" Using ALL {len(all_tickers)} tickers for optimization")
        return all_tickers
    
    elif config.OPTUNA_TICKER_SELECTION == "selected":
        if selected_tickers_global is None:
            print(" Loading selected tickers...")
            selected_tickers_global = load_selected_tickers()
        
        if not selected_tickers_global:
            print("  No selected tickers found, falling back to all tickers")
            return all_tickers
        
        # Verify selected tickers exist in data
        available_selected = [t for t in selected_tickers_global if t in all_tickers]
        missing_selected = [t for t in selected_tickers_global if t not in all_tickers]
        
        if missing_selected:
            print(f"  {len(missing_selected)} selected tickers not found in data: {missing_selected}")
        
        if not available_selected:
            print("  None of the selected tickers found in data, falling back to all tickers")
            return all_tickers
        
        print(f" Using {len(available_selected)} SELECTED tickers for optimization: {available_selected}")
        return available_selected
    
    elif config.OPTUNA_TICKER_SELECTION == "random":
        np.random.seed(config.RANDOM_STATE)
        n_random = min(config.RANDOM_TICKER_COUNT, len(all_tickers))
        random_tickers = np.random.choice(all_tickers, size=n_random, replace=False).tolist()
        print(f" Using {n_random} RANDOM tickers for optimization: {random_tickers}")
        return random_tickers
    
    else:
        print(f"  Unknown ticker selection mode: {config.OPTUNA_TICKER_SELECTION}, using all tickers")
        return all_tickers

def set_ticker_selection_mode(mode: str, random_count: int = 10):
    """Set ticker selection mode for optimization."""
    valid_modes = ["all", "selected", "random"]
    
    if mode not in valid_modes:
        print(f"✗ Invalid mode '{mode}'. Valid modes: {valid_modes}")
        return
    
    config.OPTUNA_TICKER_SELECTION = mode
    if mode == "random":
        config.RANDOM_TICKER_COUNT = random_count
    
    print(f"✓ Ticker selection mode set to: {mode}")
    if mode == "random":
        print(f"   Random ticker count: {random_count}")
    elif mode == "selected":
        print(f"   Selected tickers file: {config.SELECTED_TICKERS_FILE}")

#------------------------------------------------
# ENHANCED DATA PREPROCESSING
#------------------------------------------------
def load_and_preprocess_data(csv_file: str) -> pd.DataFrame:
    """
    Loads and preprocesses the new CSV format with direct columns.
    Expected columns: ['Date', 'Symbol', 'Close', 'Volume', 'MACD', 'RSI', 'CCI', 'ADX', 'Sentiment_Label']
    """
    logger.info(f"Loading data from {csv_file}...")
    
    # Load the CSV
    df = pd.read_csv(csv_file, parse_dates=['Date'])
    
    # Validate expected columns
    expected_base_cols = ['Date', 'Symbol', 'Close', 'Volume', 'MACD', 'RSI', 'CCI', 'ADX']
    missing_cols = [col for col in expected_base_cols if col not in df.columns]
    
    if missing_cols:
        logger.warning(f"Missing expected columns: {missing_cols}")
        print(f" Warning: Missing columns {missing_cols}")
        print(f"Available columns: {list(df.columns)}")
    
    # Rename Symbol to Ticker for consistency
    if 'Symbol' in df.columns:
        df = df.rename(columns={'Symbol': 'Ticker'})
    
    # Drop rows with missing Close prices (target variable)
    df_clean = df.dropna(subset=['Close']).copy()
    
    # Create time index
    df_clean['time_idx'] = (df_clean['Date'] - df_clean['Date'].min()).dt.days
    
    # Create cyclical time features
    df_clean['month'] = df_clean['Date'].dt.month
    df_clean['month_sin'] = np.sin(2 * np.pi * df_clean['month'] / 12)
    df_clean['month_cos'] = np.cos(2 * np.pi * df_clean['month'] / 12)
    
    # Sort by ticker and time
    df_clean = df_clean.sort_values(['Ticker', 'time_idx'])
    
    # Handle missing values in technical indicators
    technical_indicators = ['Volume', 'MACD', 'RSI', 'CCI', 'ADX']
    available_indicators = [col for col in technical_indicators if col in df_clean.columns]
    
    # Forward fill missing values within each ticker group
    for indicator in available_indicators:
        df_clean[indicator] = df_clean.groupby('Ticker')[indicator].fillna(method='ffill')
        df_clean[indicator] = df_clean.groupby('Ticker')[indicator].fillna(method='bfill')
    
    # Handle Sentiment_Label if present
    if 'Sentiment_Label' in df_clean.columns:
        sentiment_fill_value = df_clean['Sentiment_Label'].median()
        df_clean['Sentiment_Label'] = df_clean['Sentiment_Label'].fillna(sentiment_fill_value)
    
    logger.info("Data preprocessing complete.")
    logger.info(f"Final dataset shape: {df_clean.shape}")
    logger.info(f"Available columns: {list(df_clean.columns)}")
    
    return df_clean

def preview_selected_data():
    """Preview data for selected tickers only."""
    global df_clean_global, selected_tickers_global
    
    if df_clean_global is None:
        print(" Loading data first...")
        if not preview_data():
            return False
    
    optimization_tickers = get_optimization_tickers(df_clean_global)
    
    if not optimization_tickers:
        print("✗ No optimization tickers available!")
        return False
    
    # Filter data for selected tickers
    selected_df = df_clean_global[df_clean_global['Ticker'].isin(optimization_tickers)].copy()
    
    print("\n" + "="*70)
    print(" SELECTED TICKERS DATA ANALYSIS")
    print("="*70)
    
    print(f"Selection Mode: {config.OPTUNA_TICKER_SELECTION.upper()}")
    print(f"Selected Tickers: {len(optimization_tickers)}")
    print(f"Selected Data Shape: {selected_df.shape}")
    print(f"Date Range: {selected_df['Date'].min()} to {selected_df['Date'].max()}")
    
    # Per-ticker statistics
    print(f"\nPER-TICKER STATISTICS:")
    ticker_stats = selected_df.groupby('Ticker').agg({
        'Date': ['count', 'min', 'max'],
        'Close': ['mean', 'min', 'max', 'std']
    }).round(2)
    
    ticker_stats.columns = ['Count', 'Start_Date', 'End_Date', 'Avg_Price', 'Min_Price', 'Max_Price', 'Price_Std']
    print(ticker_stats.to_string())
    
    # Check data sufficiency for optimization
    min_required = config.MAX_ENCODER_LENGTH + config.MAX_PREDICTION_LENGTH
    insufficient_tickers = []
    
    print(f"\nDATA SUFFICIENCY CHECK:")
    print(f"Required minimum observations per ticker: {min_required}")
    
    for ticker in optimization_tickers:
        ticker_data = selected_df[selected_df['Ticker'] == ticker]
        if len(ticker_data) < min_required:
            insufficient_tickers.append((ticker, len(ticker_data)))
    
    if insufficient_tickers:
        print(f"  {len(insufficient_tickers)} tickers have insufficient data:")
        for ticker, count in insufficient_tickers:
            print(f"   {ticker}: {count} obs (need {min_required-count} more)")
    else:
        print("✓ All selected tickers have sufficient data for optimization!")
    
    print("="*70)
    return True

def preview_data():
    """Enhanced data preview with comprehensive analysis."""
    global df_clean_global
    
    print("COMPREHENSIVE DATA ANALYSIS")
    print("=" * 60)
    
    try:
        df_clean_global = load_and_preprocess_data(config.CSV_FILE)
        
        print(f"✓ Data loaded successfully!")
        print(f"Dataset Shape: {df_clean_global.shape}")
        print(f"Date Range: {df_clean_global['Date'].min()} to {df_clean_global['Date'].max()}")
        print(f"Unique Tickers: {df_clean_global['Ticker'].nunique()}")
        print(f"Tickers: {sorted(df_clean_global['Ticker'].unique())}")
        
        # Show available columns and categorize them
        all_cols = df_clean_global.columns.tolist()
        
        # Categorize columns
        target_col = ['Close']
        time_cols = ['Date', 'time_idx', 'month', 'month_sin', 'month_cos', 'day_of_week']
        id_cols = ['Ticker']
        technical_cols = [col for col in ['Volume', 'MACD', 'RSI', 'CCI', 'ADX'] if col in all_cols]
        sentiment_cols = [col for col in ['Sentiment_Label'] if col in all_cols]
        
        print(f"\nCOLUMN ANALYSIS:")
        print(f"   Target: {target_col}")
        print(f"   ID Columns: {id_cols}")
        print(f"   Time Features: {[col for col in time_cols if col in all_cols]}")
        print(f"   Technical Indicators: {technical_cols}")
        if sentiment_cols:
            print(f"   Sentiment Features: {sentiment_cols}")
        
        # Data quality analysis
        print(f"\nDATA QUALITY:")
        missing_data = df_clean_global.isnull().sum()
        if missing_data.sum() > 0:
            print("   Missing Data Found:")
            for col, missing in missing_data[missing_data > 0].items():
                percentage = (missing / len(df_clean_global)) * 100
                print(f"      {col}: {missing} missing ({percentage:.1f}%)")
        else:
            print("   ✓ No missing data found!")
        
        # Show sample data
        print(f"\nSAMPLE DATA (first 5 rows):")
        display_cols = ['Date', 'Ticker', 'Close'] + technical_cols[:3]
        if sentiment_cols:
            display_cols.extend(sentiment_cols)
        sample_data = df_clean_global[display_cols].head()
        print(sample_data.to_string(index=False))
        
        # Statistical analysis
        print(f"\nCLOSE PRICE STATISTICS BY TICKER:")
        close_stats = df_clean_global.groupby('Ticker')['Close'].agg([
            ('Count', 'count'),
            ('Mean', lambda x: f"{x.mean():.2f}"),
            ('Min', lambda x: f"{x.min():.2f}"),
            ('Max', lambda x: f"{x.max():.2f}"),
            ('Std', lambda x: f"{x.std():.2f}")
        ])
        print(close_stats.head(10).to_string())  # Show first 10 tickers
        
        if len(close_stats) > 10:
            print(f"... and {len(close_stats) - 10} more tickers")
        
        # Check minimum data requirements
        min_required = config.MAX_ENCODER_LENGTH + config.MAX_PREDICTION_LENGTH
        date_counts = df_clean_global.groupby('Ticker')['Date'].count()
        tickers_insufficient = date_counts[date_counts < min_required]
        
        print(f"\nDATA SUFFICIENCY:")
        print(f"   Required minimum per ticker: {min_required} observations")
        print(f"   Sufficient tickers: {len(date_counts) - len(tickers_insufficient)}")
        
        if len(tickers_insufficient) > 0:
            print(f"  {len(tickers_insufficient)} tickers have insufficient data")
            print(f"   Consider excluding these from optimization")
        else:
            print("   ✓ All tickers have adequate data!")
        
        print(f"\nOPTUNA CONFIGURATION:")
        print(f"   Selection Mode: {config.OPTUNA_TICKER_SELECTION}")
        print(f"   Selected Tickers File: {config.SELECTED_TICKERS_FILE}")
        
        print(f"\nNext steps:")
        print("   • preview_selected_data() - Preview optimization data")
        print("   • run_optimization() - Start hyperparameter optimization")
        
    except FileNotFoundError:
        print(f"✗ File '{config.CSV_FILE}' not found!")
        print("   Please make sure your CSV file is in the correct location.")
        return False
    except Exception as e:
        print(f"✗ Error loading data: {str(e)}")
        return False
    
    print("=" * 60)
    return True

#------------------------------------------------
# ENHANCED MODEL PIPELINE WITH TICKER FILTERING
#------------------------------------------------
def create_tft_datasets(df: pd.DataFrame, use_selected_tickers: bool = True):
    """Creates training and validation TimeSeriesDataSet objects with optional ticker filtering."""
    
    if use_selected_tickers:
        optimization_tickers = get_optimization_tickers(df)
        if optimization_tickers:
            df = df[df['Ticker'].isin(optimization_tickers)].copy()
            print(f" Using {len(optimization_tickers)} tickers for dataset creation")
        else:
            print("  No optimization tickers found, using all data")
    
    training_cutoff = df['time_idx'].max() - config.MAX_PREDICTION_LENGTH

    # Determine available features
    available_indicators = [col for col in ['Volume', 'MACD', 'RSI', 'CCI', 'ADX'] if col in df.columns]
    time_varying_unknown = available_indicators.copy()
    
    # Add sentiment if available
    if 'Sentiment_Label' in df.columns:
        time_varying_unknown.append('Sentiment_Label')
    
    print(f" Creating datasets with features:")
    print(f"    Target: Close")
    print(f"    Group ID: Ticker")
    print(f"    Time features: time_idx, month_sin, month_cos")
    print(f"    Technical indicators: {available_indicators}")
    if 'Sentiment_Label' in df.columns:
        print(f"    Sentiment: Sentiment_Label")

    training_data = TimeSeriesDataSet(
        df[lambda x: x.time_idx <= training_cutoff],
        time_idx="time_idx",
        target="Close",
        group_ids=["Ticker"],
        max_encoder_length=config.MAX_ENCODER_LENGTH,
        max_prediction_length=config.MAX_PREDICTION_LENGTH,
        static_categoricals=["Ticker"],
        time_varying_known_reals=["time_idx", "month_sin", "month_cos"],
        time_varying_unknown_reals=time_varying_unknown,
        allow_missing_timesteps=True
    )
    
    validation_data = TimeSeriesDataSet.from_dataset(
        training_data, df, predict=True, stop_randomization=True
    )
    
    return training_data, validation_data

# optuna objective fucntion
def objective(trial: optuna.Trial, training_data: TimeSeriesDataSet, val_dataloader) -> float:
    """Enhanced objective function with more hyperparameters."""
    params = {
        "hidden_size": trial.suggest_categorical("hidden_size", [32, 64, 128, 256]),
        "attention_head_size": trial.suggest_categorical("attention_head_size", [1, 2, 4]),
        "dropout": trial.suggest_float("dropout", 0.1, 0.4),
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True),
        "hidden_continuous_size": trial.suggest_categorical("hidden_continuous_size", [8, 16, 32]),
        "lstm_layers": trial.suggest_int("lstm_layers", 1, 3),
    }

    model = TemporalFusionTransformer.from_dataset(
        training_data,
        loss=MAE(),
        **params
    )
    
    train_dataloader = training_data.to_dataloader(train=True, batch_size=config.BATCH_SIZE, num_workers=0)

    trainer = pl.Trainer(
        max_epochs=12,
        gpus=1 if torch.cuda.is_available() else 0,
        callbacks=[EarlyStopping(monitor="val_loss", patience=config.EARLY_STOPPING_PATIENCE, mode="min")],
        logger=False,
        enable_progress_bar=True,
        enable_model_summary=False,
        gradient_clip_val=0.1
    )

    try:
        trainer.fit(model, train_dataloader, val_dataloader)
        val_loss = trainer.callback_metrics["val_loss"].item()
        
        print(f"Trial {trial.number}: Loss = {val_loss:.4f}, Params = {params}")
        return val_loss
        
    except Exception as e:
        print(f"Trial {trial.number} failed: {str(e)}")
        return float('inf')

def run_optimization():
    """Enhanced optimization phase with ticker filtering."""
    global training_data_global, validation_data_global, df_clean_global
    
    print(" STARTING ENHANCED HYPERPARAMETER OPTIMIZATION")
    print("=" * 70)
    
    # Load data if not already loaded
    if df_clean_global is None:
        print(" Loading data first...")
        if not preview_data():
            return None, None
    
    # Show selected ticker info
    optimization_tickers = get_optimization_tickers(df_clean_global)
    if not optimization_tickers:
        print("✗ No tickers available for optimization!")
        return None, None
    
    pl.seed_everything(config.RANDOM_STATE, workers=True)
    
    # Create datasets with selected tickers only
    print(" Creating optimization datasets...")
    training_data_global, validation_data_global = create_tft_datasets(
        df_clean_global, 
        use_selected_tickers=True
    )
    
    val_dataloader = validation_data_global.to_dataloader(
        train=False, 
        batch_size=config.BATCH_SIZE, 
        num_workers=0
    )
    
    print(f"    Training samples: {len(training_data_global)}")
    print(f"    Validation samples: {len(validation_data_global)}")
    
    print(f"\n Starting optimization with {config.N_TRIALS} trials...")
    print("=" * 70)

    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    )
    
    study.optimize(
        lambda trial: objective(trial, training_data_global, val_dataloader), 
        n_trials=config.N_TRIALS
    )

    best_params = analyze_optimization_results(study)
    
    return study, best_params

def analyze_optimization_results(study):
    """Enhanced optimization analysis with detailed insights."""
    global optimization_results
    
    print("\n" + "="*80)
    print(" HYPERPARAMETER OPTIMIZATION COMPLETED")
    print("="*80)
    
    # Basic study information
    print(f"Study completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Number of trials: {len(study.trials)}")
    
    completed_trials = [t for t in study.trials if t.state.name == 'COMPLETE']
    failed_trials = [t for t in study.trials if t.state.name == 'FAIL']
    
    print(f"   Completed trials: {len(completed_trials)}")
    print(f"   Failed trials: {len(failed_trials)}")
    print(f"   Success rate: {len(completed_trials)/len(study.trials)*100:.1f}%")
    
    if study.best_trial is None:
        print("✗ No successful trials found!")
        return None
    
    # Best trial information
    print(f"\n BEST TRIAL RESULTS:")
    print(f"   Trial Number: {study.best_trial.number}")
    print(f"   Best Validation Loss: {study.best_trial.value:.6f}")
    print(f"   Best Parameters:")
    for param, value in study.best_trial.params.items():
        print(f"      {param}: {value}")
    
    # Trial performance analysis
    if len(completed_trials) > 1:
        losses = [t.value for t in completed_trials]
        print(f"\n TRIAL PERFORMANCE ANALYSIS:")
        print(f"   Best Loss: {min(losses):.6f}")
        print(f"   Worst Loss: {max(losses):.6f}")
        print(f"   Average Loss: {np.mean(losses):.6f}")
        print(f"   Std Dev: {np.std(losses):.6f}")
        
        improvement = ((max(losses) - min(losses)) / max(losses)) * 100
        print(f"   Improvement: {improvement:.2f}% from worst to best")
    
    # Store results globally
    optimization_results = {
        'study': study,
        'best_params': study.best_trial.params,
        'best_loss': study.best_trial.value,
        'best_trial_number': study.best_trial.number,
        'total_trials': len(study.trials),
        'completed_trials': len(completed_trials),
        'success_rate': len(completed_trials)/len(study.trials),
        'timestamp': datetime.now().isoformat(),
        'optimization_tickers': get_optimization_tickers(df_clean_global)
    }
    
    print(f"\n NEXT STEPS:")
    print("   • show_best_params() - View best parameters again")
    print("   • train_final_model_with_best_params() - Train with best params")
    print("   • train_final_model_with_all_data() - Train on ALL tickers using best params")
    
    print("="*80)
    
    return study.best_trial.params

def show_best_params():
    """Display the best parameters from optimization."""
    global optimization_results
    
    if not optimization_results:
        print("  No optimization results found. Run run_optimization() first.")
        return None
    
    print(" BEST HYPERPARAMETERS")
    print("=" * 50)
    print(f"Best Loss: {optimization_results['best_loss']:.6f}")
    print(f"Trial Number: {optimization_results['best_trial_number']}")
    print(f"Optimization Tickers: {len(optimization_results.get('optimization_tickers', []))}")
    print("\nBest Parameters:")
    
    for param, value in optimization_results['best_params'].items():
        print(f"   {param}: {value}")
    
    print("=" * 50)
    return optimization_results['best_params']

def quick_train_selected(n_trials=5, max_epochs=25):
    """Quick training pipeline using selected tickers."""
    print(" QUICK TRAINING PIPELINE (SELECTED TICKERS)")
    print("=" * 60)
    
    original_trials = config.N_TRIALS
    config.N_TRIALS = n_trials
    
    try:
        print(" Step 1: Loading and previewing data...")
        if not preview_data():
            return None, None
        
        print(" Step 2: Loading selected tickers...")
        load_selected_tickers()
        
        print(" Step 3: Previewing selected data...")
        preview_selected_data()
        
        print(f"\n Step 4: Quick optimization ({n_trials} trials)...")
        study, best_params = run_optimization()
        
        if best_params:
            print(f"\n QUICK TRAINING WITH SELECTED TICKERS COMPLETED!")
            print(f"   Use train_final_model_with_best_params() to train final model")
            print(f"   Or train_final_model_with_all_data() to train on all tickers")
            
            return study, best_params
        
        return None, None
        
    finally:
        config.N_TRIALS = original_trials

#------------------------------------------------
# ENHANCED FINAL MODEL TRAINING
#------------------------------------------------
def train_final_model_with_best_params(use_all_data: bool = False, max_epochs: int = 50):
    """Train final model with best parameters, option to use all data or selected tickers."""
    global optimization_results, df_clean_global
    
    if not optimization_results or 'best_params' not in optimization_results:
        print("  No best parameters found. Run optimization first!")
        return None
    
    print(" TRAINING FINAL MODEL WITH BEST PARAMETERS")
    print("=" * 70)
    
    best_params = optimization_results['best_params']
    print("Best Parameters:")
    for param, value in best_params.items():
        print(f"   {param}: {value}")
    
    # Determine data usage
    data_scope = "ALL TICKERS" if use_all_data else "SELECTED TICKERS"
    print(f"\nTraining Scope: {data_scope}")
    
    # Create datasets
    print(" Creating final training datasets...")
    training_data, validation_data = create_tft_datasets(
        df_clean_global, 
        use_selected_tickers=not use_all_data
    )
    
    print(f"    Training samples: {len(training_data)}")
    print(f"    Validation samples: {len(validation_data)}")
    
    # Create data loaders
    train_dataloader = training_data.to_dataloader(
        train=True, 
        batch_size=config.BATCH_SIZE, 
        num_workers=0
    )
    val_dataloader = validation_data.to_dataloader(
        train=False, 
        batch_size=config.BATCH_SIZE, 
        num_workers=0
    )
    
    # Create model with best parameters
    print(" Creating model with optimized parameters...")
    model = TemporalFusionTransformer.from_dataset(
        training_data,
        loss=MAE(),
        **best_params
    )
    
    # Set up callbacks
    checkpoint_callback = ModelCheckpoint(
        dirpath=config.CHECKPOINT_DIR,
        filename=f'tft-best-model-{datetime.now().strftime("%Y%m%d_%H%M%S")}',
        monitor='val_loss',
        mode='min',
        save_top_k=1
    )
    
    early_stop_callback = EarlyStopping(
        monitor="val_loss",
        patience=config.EARLY_STOPPING_PATIENCE,
        verbose=True,
        mode="min"
    )
    
    # Set up trainer
    trainer = pl.Trainer(
        max_epochs=max_epochs,
        gpus=1 if torch.cuda.is_available() else 0,
        callbacks=[checkpoint_callback, early_stop_callback],
        gradient_clip_val=0.1,
        enable_progress_bar=True
    )
    
    print(f" Starting training ({max_epochs} max epochs)...")
    print("=" * 70)
    
    # Train the model
    trainer.fit(model, train_dataloader, val_dataloader)
    
    # Load best model
    best_model = TemporalFusionTransformer.load_from_checkpoint(checkpoint_callback.best_model_path)
    
    print("\n FINAL MODEL TRAINING COMPLETED!")
    print("=" * 70)
    print(f"Best model saved at: {checkpoint_callback.best_model_path}")
    print(f"Final validation loss: {trainer.callback_metrics.get('val_loss', 'N/A')}")
    
    print("\n NEXT STEPS:")
    print("   • evaluate_model(model) - Comprehensive evaluation")
    print("   • plot_predictions(model) - Visualize predictions")
    print("   • export_model(model, 'my_model.pkl') - Save model")
    
    return best_model

def train_final_model_with_all_data(max_epochs: int = 50):
    """Train final model using ALL tickers with best parameters."""
    return train_final_model_with_best_params(use_all_data=True, max_epochs=max_epochs)

def train_final_model_with_custom_params(params: Dict[str, Any], use_all_data: bool = False, max_epochs: int = 50):
    """Train model with custom parameters."""
    global df_clean_global
    
    if df_clean_global is None:
        print("  Data not loaded. Run preview_data() first!")
        return None
    
    print(" TRAINING MODEL WITH CUSTOM PARAMETERS")
    print("=" * 70)
    
    print("Custom Parameters:")
    for param, value in params.items():
        print(f"   {param}: {value}")
    
    data_scope = "ALL TICKERS" if use_all_data else "SELECTED TICKERS"
    print(f"\nTraining Scope: {data_scope}")
    
    # Create datasets
    training_data, validation_data = create_tft_datasets(
        df_clean_global, 
        use_selected_tickers=not use_all_data
    )
    
    # Create data loaders
    train_dataloader = training_data.to_dataloader(train=True, batch_size=config.BATCH_SIZE, num_workers=0)
    val_dataloader = validation_data.to_dataloader(train=False, batch_size=config.BATCH_SIZE, num_workers=0)
    
    # Create model
    model = TemporalFusionTransformer.from_dataset(
        training_data,
        loss=MAE(),
        **params
    )
    
    # Set up trainer
    trainer = pl.Trainer(
        max_epochs=max_epochs,
        gpus=1 if torch.cuda.is_available() else 0,
        callbacks=[EarlyStopping(monitor="val_loss", patience=config.EARLY_STOPPING_PATIENCE, mode="min")],
        gradient_clip_val=0.1
    )
    
    # Train
    trainer.fit(model, train_dataloader, val_dataloader)
    
    print(" Custom model training completed!")
    return model

#------------------------------------------------
# MODEL EVALUATION AND VISUALIZATION
#------------------------------------------------
def evaluate_model(model, use_selected_data: bool = True):
    """Comprehensive model evaluation."""
    global df_clean_global, validation_data_global
    
    if model is None:
        print("  No model provided!")
        return None
    
    if validation_data_global is None:
        print("  No validation data found. Train a model first!")
        return None
    
    print(" COMPREHENSIVE MODEL EVALUATION")
    print("=" * 60)
    
    # Create validation dataloader
    val_dataloader = validation_data_global.to_dataloader(
        train=False, 
        batch_size=config.BATCH_SIZE, 
        num_workers=0
    )
    
    # Get predictions
    print(" Generating predictions...")
    predictions = model.predict(val_dataloader, return_y=True)
    
    # Calculate metrics
    mae = MAE()
    rmse = RMSE()
    mape = MAPE()
    
    mae_score = mae(predictions.output, predictions.y)
    rmse_score = rmse(predictions.output, predictions.y)
    mape_score = mape(predictions.output, predictions.y)
    
    print(f"\n OVERALL METRICS:")
    print(f"   MAE (Mean Absolute Error): {mae_score:.4f}")
    print(f"   RMSE (Root Mean Squared Error): {rmse_score:.4f}")
    print(f"   MAPE (Mean Absolute Percentage Error): {mape_score:.4f}%")
    
    # Per-ticker analysis if multiple tickers
    if hasattr(validation_data_global, 'data'):
        unique_tickers = validation_data_global.data['Ticker'].unique()
        if len(unique_tickers) > 1:
            print(f"\n PER-TICKER PERFORMANCE:")
            print(f"   Evaluated on {len(unique_tickers)} tickers")
            
            # You can add more detailed per-ticker analysis here
            ticker_sample = unique_tickers[:5] if len(unique_tickers) > 5 else unique_tickers
            print(f"   Sample tickers: {list(ticker_sample)}")
    
    # Feature importance
    print(f"\n FEATURE IMPORTANCE:")
    try:
        interpretation = model.interpret_output(predictions.output, reduction="sum")
        
        if hasattr(interpretation, 'attention'):
            attention_scores = interpretation.attention.mean(0).cpu().numpy()
            feature_names = model.hparams.time_varying_unknown_reals
            
            if len(attention_scores) == len(feature_names):
                feature_importance = list(zip(feature_names, attention_scores))
                feature_importance.sort(key=lambda x: x[1], reverse=True)
                
                print("   Top features by attention:")
                for feature, score in feature_importance[:5]:
                    print(f"      {feature}: {score:.4f}")
            else:
                print("   Feature importance analysis not available")
        else:
            print("   Attention scores not available")
    except Exception as e:
        print(f"   Feature importance analysis failed: {str(e)}")
    
    evaluation_results = {
        'mae': mae_score,
        'rmse': rmse_score,
        'mape': mape_score,
        'predictions': predictions,
        'n_tickers': len(unique_tickers) if 'unique_tickers' in locals() else 1,
        'timestamp': datetime.now().isoformat()
    }
    
    print("=" * 60)
    return evaluation_results

def plot_predictions(model, n_samples: int = 5):
    """Enhanced prediction visualization."""
    global validation_data_global
    
    if model is None or validation_data_global is None:
        print("  Model or validation data not available!")
        return
    
    print(f" GENERATING PREDICTION PLOTS ({n_samples} samples)")
    print("=" * 50)
    
    # Create validation dataloader
    val_dataloader = validation_data_global.to_dataloader(
        train=False, 
        batch_size=config.BATCH_SIZE, 
        num_workers=0
    )
    
    # Get predictions
    predictions = model.predict(val_dataloader, return_y=True)
    
    # Create plots
    fig, axes = plt.subplots(n_samples, 1, figsize=(15, 4*n_samples))
    if n_samples == 1:
        axes = [axes]
    
    for i in range(min(n_samples, len(predictions.output))):
        ax = axes[i]
        
        # Get actual and predicted values
        actual = predictions.y[i].cpu().numpy()
        predicted = predictions.output[i].cpu().numpy()
        
        # Plot
        time_steps = range(len(actual))
        ax.plot(time_steps, actual, label='Actual', color='blue', linewidth=2)
        ax.plot(time_steps, predicted, label='Predicted', color='red', linewidth=2, linestyle='--')
        
        # Calculate sample metrics
        sample_mae = np.mean(np.abs(actual - predicted))
        sample_mape = np.mean(np.abs((actual - predicted) / actual)) * 100
        
        ax.set_title(f'Sample {i+1} - MAE: {sample_mae:.4f}, MAPE: {sample_mape:.2f}%')
        ax.set_xlabel('Time Steps')
        ax.set_ylabel('Close Price')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(" Prediction plots generated successfully!")

def export_model(model, filename: str):
    """Export trained model."""
    if model is None:
        print("  No model to export!")
        return False
    
    try:
        # Save model
        model_path = Path(filename)
        torch.save(model.state_dict(), model_path)
        
        # Save metadata
        metadata = {
            'model_type': 'TemporalFusionTransformer',
            'timestamp': datetime.now().isoformat(),
            'config': {
                'max_encoder_length': config.MAX_ENCODER_LENGTH,
                'max_prediction_length': config.MAX_PREDICTION_LENGTH,
                'batch_size': config.BATCH_SIZE
            }
        }
        
        if optimization_results:
            metadata['optimization_results'] = optimization_results
        
        metadata_path = model_path.with_suffix('.json')
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f" Model exported successfully!")
        print(f"   Model: {model_path}")
        print(f"   Metadata: {metadata_path}")
        
        return True
        
    except Exception as e:
        print(f"✗ Export failed: {str(e)}")
        return False

def update_config(**kwargs):
    """Update configuration parameters."""
    global config
    
    print("  UPDATING CONFIGURATION")
    print("=" * 40)
    
    for key, value in kwargs.items():
        if hasattr(config, key.upper()):
            setattr(config, key.upper(), value)
            print(f"   ✓ {key.upper()}: {value}")
        else:
            print(f"     Unknown config key: {key}")
    
    print("=" * 40)

def show_data_info():
    """Show information about the expected data format and ticker selection."""
    print(" DATA FORMAT & TICKER SELECTION INFORMATION")
    print("=" * 60)
    print("Expected CSV columns:")
    print("    Date - Date column (YYYY-MM-DD format)")
    print("    Symbol - Stock ticker symbol")  
    print("    Close - Closing price (target variable)")
    print("    Volume - Trading volume")
    print("    MACD - Moving Average Convergence Divergence")
    print("    RSI - Relative Strength Index")
    print("    CCI - Commodity Channel Index") 
    print("    ADX - Average Directional Index")
    print("    Sentiment_Label - Sentiment indicator (optional)")
    
    print(f"\n TICKER SELECTION:")
    print(f"    Selected tickers file: {config.SELECTED_TICKERS_FILE}")
    print(f"    Current selection mode: {config.OPTUNA_TICKER_SELECTION}")
    
    print(f"\n SELECTION MODES:")
    print(f"    'selected' - Use tickers from {config.SELECTED_TICKERS_FILE}")
    print(f"    'all' - Use all available tickers")
    print(f"    'random' - Use random subset of tickers")
    
    print(f"\n  CURRENT CONFIG:")
    print(f"    CSV file: {config.CSV_FILE}")
    print(f"    Encoder length: {config.MAX_ENCODER_LENGTH}")
    print(f"    Prediction length: {config.MAX_PREDICTION_LENGTH}")
    print(f"    Batch size: {config.BATCH_SIZE}")
    print(f"    Optimization trials: {config.N_TRIALS}")
    print("=" * 60)

# Final setup message
print("\n ENHANCED PIPELINE READY!")
print("=" * 70)
print("RECOMMENDED WORKFLOW:")
print("1.  load_selected_tickers() - Load your ticker selection")
print("2.  preview_data() - Analyze your complete data")
print("3.  preview_selected_data() - Analyze selected ticker data")
print("4.  run_optimization() - Find best hyperparameters (uses selected tickers)")
print("5.  train_final_model_with_best_params() - Train on selected tickers")
print("6.  train_final_model_with_all_data() - Train on ALL tickers")
print("7.  evaluate_model(model) - Comprehensive evaluation")
print("8.  plot_predictions(model) - Visualize results")

print(f"\nQUICK OPTIONS:")
print("•  quick_train_selected() - Fast end-to-end with selected tickers")
print("•  set_ticker_selection_mode('selected'/'all'/'random') - Change mode")
print("•  show_data_info() - View data format & selection info")
print("•  update_config(csv_file='your_file.csv') - Change settings")
print("=" * 70)


 ENHANCED PIPELINE - TICKER SELECTION FOR OPTUNA
Available functions:
1. load_selected_tickers() - Load ticker list from file
2. preview_data() - Preview and analyze your data
3. preview_selected_data() - Preview data for selected tickers only
4. run_optimization() - Start hyperparameter optimization (uses selected tickers)
5. show_best_params() - View best parameters
6. train_final_model_with_best_params() - Train final model (can use all data)
7. train_final_model_with_custom_params(params) - Train with custom params
8. plot_predictions(model, n_samples=5) - Visualize predictions
9. evaluate_model(model) - Comprehensive model evaluation
10. export_model(model, filename) - Export trained model
11. quick_train_selected() - Fast pipeline with selected tickers
12. set_ticker_selection_mode(mode) - Change ticker selection mode

 ENHANCED PIPELINE READY!
RECOMMENDED WORKFLOW:
1.  load_selected_tickers() - Load your ticker selection
2.  preview_data() - Analyze your complete data
3.  previ

In [5]:
run_optimization()

2025-08-06 14:05:58,228 - INFO - Loading data from sentiment_indicator_stock.csv...


 STARTING ENHANCED HYPERPARAMETER OPTIMIZATION
 Loading data first...
COMPREHENSIVE DATA ANALYSIS


2025-08-06 14:05:58,570 - INFO - Data preprocessing complete.
2025-08-06 14:05:58,571 - INFO - Final dataset shape: (109100, 13)
2025-08-06 14:05:58,572 - INFO - Available columns: ['Date', 'Ticker', 'Close', 'Volume', 'MACD', 'RSI', 'CCI', 'ADX', 'Sentiment_Label', 'time_idx', 'month', 'month_sin', 'month_cos']
Global seed set to 42


✓ Data loaded successfully!
Dataset Shape: (109100, 13)
Date Range: 2003-04-01 00:00:00 to 2025-03-28 00:00:00
Unique Tickers: 20
Tickers: ['ACC.NS', 'AXISBANK.NS', 'BHEL.NS', 'CIPLA.NS', 'DRREDDY.NS', 'GAIL.NS', 'GRASIM.NS', 'HDFCBANK.NS', 'HINDUNILVR.NS', 'ICICIBANK.NS', 'INFY.NS', 'KOTAKBANK.NS', 'LT.NS', 'MRF.NS', 'NCC.NS', 'PNB.NS', 'RELIANCE.NS', 'SBIN.NS', 'TVSMOTOR.NS', 'WIPRO.NS']

COLUMN ANALYSIS:
   Target: ['Close']
   ID Columns: ['Ticker']
   Time Features: ['Date', 'time_idx', 'month', 'month_sin', 'month_cos']
   Technical Indicators: ['Volume', 'MACD', 'RSI', 'CCI', 'ADX']
   Sentiment Features: ['Sentiment_Label']

DATA QUALITY:
   ✓ No missing data found!

SAMPLE DATA (first 5 rows):
      Date Ticker     Close   Volume      MACD       RSI  Sentiment_Label
2003-04-01 ACC.NS 95.407578 460007.0 -1.932525 44.446821              1.0
2003-04-02 ACC.NS 96.195770 175372.0 -1.701787 47.085214              1.0
2003-04-03 ACC.NS 95.818794 402265.0 -1.531688 45.960906          

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


...

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 2050') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 11: 100%|██████████| 191/191 [00:50<00:00,  3.81it/s, loss=20.3, train_loss_step=20.50, val_loss=62.40, train_loss_epoch=20.50]

`Trainer.fit` stopped: `max_epochs=12` reached.


Epoch 11: 100%|██████████| 191/191 [00:50<00:00,  3.80it/s, loss=20.3, train_loss_step=20.50, val_loss=62.40, train_loss_epoch=20.50]
Trial 19: Loss = 62.4029, Params = {'hidden_size': 64, 'attention_head_size': 1, 'dropout': 0.34425928311874715, 'learning_rate': 3.710866515758842e-05, 'hidden_continuous_size': 8, 'lstm_layers': 2}

 HYPERPARAMETER OPTIMIZATION COMPLETED
Study completed at: 2025-08-06 17:17:06
   Number of trials: 20
   Completed trials: 20
   Failed trials: 0
   Success rate: 100.0%

 BEST TRIAL RESULTS:
   Trial Number: 10
   Best Validation Loss: 45.985760
   Best Parameters:
      hidden_size: 64
      attention_head_size: 4
      dropout: 0.3107300476565992
      learning_rate: 0.0017277011787520675
      hidden_continuous_size: 8
      lstm_layers: 1

 TRIAL PERFORMANCE ANALYSIS:
   Best Loss: 45.985760
   Worst Loss: 83.077148
   Average Loss: 56.299057
   Std Dev: 9.118611
   Improvement: 44.65% from worst to best
 Using 8 SELECTED tickers for optimization: ['T

(<optuna.study.study.Study at 0x267978ab760>,
 {'hidden_size': 64,
  'attention_head_size': 4,
  'dropout': 0.3107300476565992,
  'learning_rate': 0.0017277011787520675,
  'hidden_continuous_size': 8,
  'lstm_layers': 1})

In [6]:
show_best_params()

 BEST HYPERPARAMETERS
Best Loss: 45.985760
Trial Number: 10
Optimization Tickers: 8

Best Parameters:
   hidden_size: 64
   attention_head_size: 4
   dropout: 0.3107300476565992
   learning_rate: 0.0017277011787520675
   hidden_continuous_size: 8
   lstm_layers: 1


{'hidden_size': 64,
 'attention_head_size': 4,
 'dropout': 0.3107300476565992,
 'learning_rate': 0.0017277011787520675,
 'hidden_continuous_size': 8,
 'lstm_layers': 1}

In [ ]:
train_final_model_with_best_params()     # best params on selected tickers

 TRAINING FINAL MODEL WITH BEST PARAMETERS
Best Parameters:
   hidden_size: 64
   attention_head_size: 4
   dropout: 0.3107300476565992
   learning_rate: 0.0017277011787520675
   hidden_continuous_size: 8
   lstm_layers: 1

Training Scope: SELECTED TICKERS
 Creating final training datasets...
 Using 8 SELECTED tickers for optimization: ['TVSMOTOR.NS', 'HDFCBANK.NS', 'INFY.NS', 'RELIANCE.NS', 'NCC.NS', 'DRREDDY.NS', 'SBIN.NS', 'ICICIBANK.NS']
 Using 8 tickers for dataset creation
 Creating datasets with features:
    Target: Close
    Group ID: Ticker
    Time features: time_idx, month_sin, month_cos
    Technical indicators: ['Volume', 'MACD', 'RSI', 'CCI', 'ADX']
    Sentiment: Sentiment_Label
    Training samples: 24440
    Validation samples: 8
 Creating model with optimized parameters...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 2050') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


 Starting training (50 max epochs)...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | MAE                             | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 40    
3  | prescalers                         | ModuleDict                      | 144   
4  | static_variable_selection          | VariableSelectionNetwork        | 192   
5  | encoder_variable_selection         | VariableSelectionNetwork        | 16.2 K
6  | decoder_variable_selection         | VariableSelectionNetwork        | 5.2 K 
7  | static_context_variable_selection  | GatedResidualNetwork            | 16.8 K
8  | static_context_initial_hidden_lstm | GatedResidualNetwork            | 16.8 K
9  | static_context_initial_cell_lstm 

Epoch 0: 100%|██████████| 191/191 [02:21<00:00,  1.35it/s, loss=21.2, v_num=2, train_loss_step=22.40, val_loss=66.40]

...

Metric val_loss improved by 5.119 >= min_delta = 0.0. New best score: 48.475


Epoch 12: 100%|██████████| 191/191 [00:48<00:00,  3.96it/s, loss=13.3, v_num=2, train_loss_step=13.80, val_loss=53.50, train_loss_epoch=12.90]

Monitored metric val_loss did not improve in the last 5 records. Best score: 48.475. Signaling Trainer to stop.


Epoch 12: 100%|██████████| 191/191 [00:48<00:00,  3.96it/s, loss=13.3, v_num=2, train_loss_step=13.80, val_loss=53.50, train_loss_epoch=12.80]

 FINAL MODEL TRAINING COMPLETED!
Best model saved at: checkpoints\tft-best-model-20250806_175650.ckpt
Final validation loss: 53.51689147949219

 NEXT STEPS:
   • evaluate_model(model) - Comprehensive evaluation
   • plot_predictions(model) - Visualize predictions
   • export_model(model, 'my_model.pkl') - Save model


TemporalFusionTransformer(
  (loss): MAE()
  (logging_metrics): ModuleList(
    (0): SMAPE()
    (1): MAE()
    (2): RMSE()
    (3): MAPE()
  )
  (input_embeddings): MultiEmbedding(
    (embeddings): ModuleDict(
      (Ticker): Embedding(8, 5)
    )
  )
  (prescalers): ModuleDict(
    (time_idx): Linear(in_features=1, out_features=8, bias=True)
    (month_sin): Linear(in_features=1, out_features=8, bias=True)
    (month_cos): Linear(in_features=1, out_features=8, bias=True)
    (Volume): Linear(in_features=1, out_features=8, bias=True)
    (MACD): Linear(in_features=1, out_features=8, bias=True)
    (RSI): Linear(in_features=1, out_features=8, bias=True)
    (CCI): Linear(in_features=1, out_features=8, bias=True)
    (ADX): Linear(in_features=1, out_features=8, bias=True)
    (Sentiment_Label): Linear(in_features=1, out_features=8, bias=True)
  )
  (static_variable_selection): VariableSelectionNetwork(
    (single_variable_grns): ModuleDict(
      (Ticker): ResampleNorm(
        (resam

In [ ]:
train_final_model_with_all_data()        # best params on selected tickers

 TRAINING FINAL MODEL WITH BEST PARAMETERS
Best Parameters:
   hidden_size: 64
   attention_head_size: 4
   dropout: 0.3107300476565992
   learning_rate: 0.0017277011787520675
   hidden_continuous_size: 8
   lstm_layers: 1

Training Scope: ALL TICKERS
 Creating final training datasets...
 Creating datasets with features:
    Target: Close
    Group ID: Ticker
    Time features: time_idx, month_sin, month_cos
    Technical indicators: ['Volume', 'MACD', 'RSI', 'CCI', 'ADX']
    Sentiment: Sentiment_Label
    Training samples: 61100
    Validation samples: 20
 Creating model with optimized parameters...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 2050') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


 Starting training (50 max epochs)...



   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | MAE                             | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 180   
3  | prescalers                         | ModuleDict                      | 144   
4  | static_variable_selection          | VariableSelectionNetwork        | 192   
5  | encoder_variable_selection         | VariableSelectionNetwork        | 16.2 K
6  | decoder_variable_selection         | VariableSelectionNetwork        | 5.2 K 
7  | static_context_variable_selection  | GatedResidualNetwork            | 16.8 K
8  | static_context_initial_hidden_lstm | GatedResidualNetwork            | 16.8 K
9  | static_context_initial_cell_lstm   | GatedResidualNetwork            | 16.8

Epoch 0: 100%|██████████| 478/478 [02:10<00:00,  3.67it/s, loss=111, v_num=3, train_loss_step=88.20, val_loss=243.0]

...

Metric val_loss improved by 30.949 >= min_delta = 0.0. New best score: 211.600


Epoch 6: 100%|██████████| 478/478 [02:07<00:00,  3.74it/s, loss=79.6, v_num=3, train_loss_step=135.0, val_loss=228.0, train_loss_epoch=79.30]

Monitored metric val_loss did not improve in the last 5 records. Best score: 211.600. Signaling Trainer to stop.


Epoch 6: 100%|██████████| 478/478 [02:07<00:00,  3.74it/s, loss=79.6, v_num=3, train_loss_step=135.0, val_loss=228.0, train_loss_epoch=78.40]

 FINAL MODEL TRAINING COMPLETED!
Best model saved at: checkpoints\tft-best-model-20250806_185947.ckpt
Final validation loss: 228.33346557617188

 NEXT STEPS:
   • evaluate_model(model) - Comprehensive evaluation
   • plot_predictions(model) - Visualize predictions
   • export_model(model, 'my_model.pkl') - Save model


TemporalFusionTransformer(
  (loss): MAE()
  (logging_metrics): ModuleList(
    (0): SMAPE()
    (1): MAE()
    (2): RMSE()
    (3): MAPE()
  )
  (input_embeddings): MultiEmbedding(
    (embeddings): ModuleDict(
      (Ticker): Embedding(20, 9)
    )
  )
  (prescalers): ModuleDict(
    (time_idx): Linear(in_features=1, out_features=8, bias=True)
    (month_sin): Linear(in_features=1, out_features=8, bias=True)
    (month_cos): Linear(in_features=1, out_features=8, bias=True)
    (Volume): Linear(in_features=1, out_features=8, bias=True)
    (MACD): Linear(in_features=1, out_features=8, bias=True)
    (RSI): Linear(in_features=1, out_features=8, bias=True)
    (CCI): Linear(in_features=1, out_features=8, bias=True)
    (ADX): Linear(in_features=1, out_features=8, bias=True)
    (Sentiment_Label): Linear(in_features=1, out_features=8, bias=True)
  )
  (static_variable_selection): VariableSelectionNetwork(
    (single_variable_grns): ModuleDict(
      (Ticker): ResampleNorm(
        (resa

In [13]:
update_config(early_stopping_patience=10)

  UPDATING CONFIGURATION
   ✓ EARLY_STOPPING_PATIENCE: 10


In [ ]:
train_final_model_with_all_data()    # best params on selected tickers

 TRAINING FINAL MODEL WITH BEST PARAMETERS
Best Parameters:
   hidden_size: 64
   attention_head_size: 4
   dropout: 0.3107300476565992
   learning_rate: 0.0017277011787520675
   hidden_continuous_size: 8
   lstm_layers: 1

Training Scope: ALL TICKERS
 Creating final training datasets...
 Creating datasets with features:
    Target: Close
    Group ID: Ticker
    Time features: time_idx, month_sin, month_cos
    Technical indicators: ['Volume', 'MACD', 'RSI', 'CCI', 'ADX']
    Sentiment: Sentiment_Label


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 2050') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | MAE                             | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 180   
3  | prescalers     

    Training samples: 61100
    Validation samples: 20
 Creating model with optimized parameters...
 Starting training (50 max epochs)...
Epoch 0: 100%|██████████| 478/478 [02:13<00:00,  3.58it/s, loss=77, v_num=4, train_loss_step=58.00, val_loss=239.0]

Metric val_loss improved. New best score: 239.285


Epoch 1: 100%|██████████| 478/478 [01:54<00:00,  4.17it/s, loss=84.2, v_num=4, train_loss_step=143.0, val_loss=218.0, train_loss_epoch=125.0]

...

Metric val_loss improved by 29.949 >= min_delta = 0.0. New best score: 188.381


Epoch 12: 100%|██████████| 478/478 [02:07<00:00,  3.76it/s, loss=74.3, v_num=4, train_loss_step=74.10, val_loss=243.0, train_loss_epoch=74.00]

Monitored metric val_loss did not improve in the last 10 records. Best score: 188.381. Signaling Trainer to stop.


Epoch 12: 100%|██████████| 478/478 [02:07<00:00,  3.76it/s, loss=74.3, v_num=4, train_loss_step=74.10, val_loss=243.0, train_loss_epoch=74.00]

 FINAL MODEL TRAINING COMPLETED!
Best model saved at: checkpoints\tft-best-model-20250806_192143.ckpt
Final validation loss: 243.3699188232422

 NEXT STEPS:
   • evaluate_model(model) - Comprehensive evaluation
   • plot_predictions(model) - Visualize predictions
   • export_model(model, 'my_model.pkl') - Save model


TemporalFusionTransformer(
  (loss): MAE()
  (logging_metrics): ModuleList(
    (0): SMAPE()
    (1): MAE()
    (2): RMSE()
    (3): MAPE()
  )
  (input_embeddings): MultiEmbedding(
    (embeddings): ModuleDict(
      (Ticker): Embedding(20, 9)
    )
  )
  (prescalers): ModuleDict(
    (time_idx): Linear(in_features=1, out_features=8, bias=True)
    (month_sin): Linear(in_features=1, out_features=8, bias=True)
    (month_cos): Linear(in_features=1, out_features=8, bias=True)
    (Volume): Linear(in_features=1, out_features=8, bias=True)
    (MACD): Linear(in_features=1, out_features=8, bias=True)
    (RSI): Linear(in_features=1, out_features=8, bias=True)
    (CCI): Linear(in_features=1, out_features=8, bias=True)
    (ADX): Linear(in_features=1, out_features=8, bias=True)
    (Sentiment_Label): Linear(in_features=1, out_features=8, bias=True)
  )
  (static_variable_selection): VariableSelectionNetwork(
    (single_variable_grns): ModuleDict(
      (Ticker): ResampleNorm(
        (resa

In [45]:
# Enhanced final model training functions with improved learning dynamics

def train_final_model_with_best_params(use_all_data: bool = False, max_epochs: int = 100):
    """Enhanced final model training with better learning dynamics."""
    global optimization_results, df_clean_global
    
    if not optimization_results or 'best_params' not in optimization_results:
        print(" No best parameters found. Run optimization first!")
        return None
    
    print(" TRAINING FINAL MODEL WITH ENHANCED SETTINGS")
    print("=" * 70)
    
    best_params = optimization_results['best_params'].copy()
    print("Best Parameters:")
    for param, value in best_params.items():
        print(f"   {param}: {value}")
    
    # ENHANCEMENT 1: Adjust learning rate for final training
    original_lr = best_params.get('learning_rate', 1e-3)
    # Use slightly lower learning rate for final training for better convergence
    final_lr = original_lr * 0.7
    best_params['learning_rate'] = final_lr
    print(f"\n Adjusted learning rate: {original_lr:.2e} -> {final_lr:.2e}")
    
    # Determine data usage
    data_scope = "ALL TICKERS" if use_all_data else "SELECTED TICKERS"
    print(f"\nTraining Scope: {data_scope}")
    
    # Create datasets
    print(" Creating final training datasets...")
    training_data, validation_data = create_tft_datasets(
        df_clean_global, 
        use_selected_tickers=not use_all_data
    )
    
    print(f"    Training samples: {len(training_data)}")
    print(f"    Validation samples: {len(validation_data)}")
    
    # ENHANCEMENT 2: Improved batch size based on data size
    optimal_batch_size = min(config.BATCH_SIZE, len(training_data) // 10)
    optimal_batch_size = max(optimal_batch_size, 16)  # Minimum batch size
    
    print(f" Optimized batch size: {config.BATCH_SIZE} -> {optimal_batch_size}")
    
    # Create data loaders with improved settings
    train_dataloader = training_data.to_dataloader(
        train=True, 
        batch_size=optimal_batch_size, 
        num_workers=0,
        shuffle=True  # Ensure shuffling
    )
    val_dataloader = validation_data.to_dataloader(
        train=False, 
        batch_size=optimal_batch_size, 
        num_workers=0,
        shuffle=False
    )
    
    # ENHANCEMENT 3: Create model with improved loss function
    print(" Creating model with enhanced loss function...")
    
    # Use QuantileLoss for better uncertainty estimation
    from pytorch_forecasting.metrics import QuantileLoss
    
    model = TemporalFusionTransformer.from_dataset(
        training_data,
        loss=QuantileLoss([0.1, 0.5, 0.9]),  # Better than MAE for time series
        optimizer="AdamW",  # Better optimizer
        **best_params
    )
    
    # ENHANCEMENT 4: Advanced callbacks
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # More sophisticated checkpoint strategy
    checkpoint_callback = ModelCheckpoint(
        dirpath=config.CHECKPOINT_DIR,
        filename=f'tft-final-{timestamp}-{{epoch:02d}}-{{val_loss:.4f}}',
        monitor='val_loss',
        mode='min',
        save_top_k=3,  # Keep top 3 models
        save_last=True,  # Also save last model
        every_n_epochs=1
    )
    
    # More patient early stopping for final training
    early_stop_callback = EarlyStopping(
        monitor="val_loss",
        patience=15,  # More patience for final training
        verbose=True,
        mode="min",
        min_delta=1e-6,  # Smaller minimum change threshold
        check_finite=True
    )
    
    # ENHANCEMENT 5: Learning rate scheduler
    from pytorch_lightning.callbacks import LearningRateMonitor
    
    lr_monitor = LearningRateMonitor(logging_interval='epoch')
    
    # ENHANCEMENT 6: Advanced trainer configuration
    # Check if CUDA deterministic setup is available
    use_deterministic = False
    if torch.cuda.is_available():
        try:
            # Try to enable deterministic mode with proper CUDA setup
            import os
            if 'CUBLAS_WORKSPACE_CONFIG' not in os.environ:
                os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
            torch.use_deterministic_algorithms(True)
            use_deterministic = True
            print(" Deterministic mode enabled with CUDA optimization")
        except:
            print("  Deterministic mode disabled due to CUDA compatibility")
            use_deterministic = False
    else:
        use_deterministic = True  # Safe on CPU
        print(" Deterministic mode enabled (CPU)")
    
    trainer = pl.Trainer(
        max_epochs=max_epochs,
        gpus=1 if torch.cuda.is_available() else 0,
        callbacks=[checkpoint_callback, early_stop_callback, lr_monitor],
        gradient_clip_val=0.1,
        gradient_clip_algorithm="norm",
        enable_progress_bar=True,
        log_every_n_steps=10,
        val_check_interval=1.0,  # Validate every epoch
        check_val_every_n_epoch=1,
        accumulate_grad_batches=1,
        precision=16 if torch.cuda.is_available() else 32,  # Mixed precision if GPU available
        deterministic=use_deterministic,  # Only enable if properly configured
        enable_model_summary=True
    )
    
    print(f" Starting enhanced training...")
    print(f"   Max epochs: {max_epochs}")
    print(f"   Early stopping patience: 15")
    print(f"   Learning rate: {final_lr:.2e}")
    print(f"   Batch size: {optimal_batch_size}")
    print(f"   Loss function: QuantileLoss")
    print("=" * 70)
    
    # Train the model
    trainer.fit(model, train_dataloader, val_dataloader)
    
    # ENHANCEMENT 7: Load and return the best model
    try:
        best_model = TemporalFusionTransformer.load_from_checkpoint(
            checkpoint_callback.best_model_path
        )
        best_model_path = checkpoint_callback.best_model_path
    except:
        # Fallback to the current model if checkpoint loading fails
        best_model = model
        best_model_path = "current_model_in_memory"
    
    # Training summary
    final_val_loss = trainer.callback_metrics.get('val_loss', 'N/A')
    
    print("\n ENHANCED FINAL MODEL TRAINING COMPLETED!")
    print("=" * 70)
    print(f"Best model path: {best_model_path}")
    print(f"Final validation loss: {final_val_loss}")
    print(f"Training epochs completed: {trainer.current_epoch + 1}")
    
    if trainer.current_epoch >= max_epochs - 1:
        print("  Training completed full epochs - consider increasing max_epochs")
    elif trainer.interrupted:
        print("⏹  Training stopped early due to no improvement")
    
    print(f"\n TRAINING INSIGHTS:")
    if hasattr(trainer, 'callback_metrics'):
        metrics = trainer.callback_metrics
        print(f"   Final validation loss: {metrics.get('val_loss', 'N/A')}")
        if 'lr-AdamW' in metrics:
            print(f"   Final learning rate: {metrics['lr-AdamW']:.2e}")
    
    print("\n NEXT STEPS:")
    print("   • evaluate_model(model) - Comprehensive evaluation")
    print("   • plot_predictions(model) - Visualize predictions")
    print("   • export_model(model, 'final_model.pkl') - Save model")
    print("=" * 70)
    
    return best_model

def create_enhanced_trainer(max_epochs: int = 100, patience: int = 15):
    """Create trainer with enhanced settings for better convergence."""
    
    callbacks = [
        # Enhanced early stopping
        EarlyStopping(
            monitor="val_loss",
            patience=patience,
            verbose=True,
            mode="min",
            min_delta=1e-6,
            check_finite=True
        ),
        
        # Better checkpointing
        ModelCheckpoint(
            dirpath=config.CHECKPOINT_DIR,
            filename='tft-enhanced-{epoch:02d}-{val_loss:.4f}',
            monitor='val_loss',
            mode='min',
            save_top_k=3,
            save_last=True
        ),
        
        # Learning rate monitoring
        LearningRateMonitor(logging_interval='epoch')
    ]
    
    # Smart deterministic configuration
    use_deterministic = False
    if torch.cuda.is_available():
        try:
            import os
            if 'CUBLAS_WORKSPACE_CONFIG' not in os.environ:
                os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
            torch.use_deterministic_algorithms(True)
            use_deterministic = True
        except:
            use_deterministic = False
    else:
        use_deterministic = True
    
    trainer = pl.Trainer(
        max_epochs=max_epochs,
        gpus=1 if torch.cuda.is_available() else 0,
        callbacks=callbacks,
        gradient_clip_val=0.1,
        gradient_clip_algorithm="norm",
        precision=16 if torch.cuda.is_available() else 32,
        val_check_interval=1.0,
        check_val_every_n_epoch=1,
        log_every_n_steps=10,
        enable_progress_bar=True,
        deterministic=use_deterministic,
        accumulate_grad_batches=1
    )
    
    return trainer

def diagnose_training_issues(model, training_data, validation_data):
    """Diagnose potential training issues."""
    print(" DIAGNOSING TRAINING ISSUES")
    print("=" * 50)
    
    # Check data sizes
    print(f"Training samples: {len(training_data)}")
    print(f"Validation samples: {len(validation_data)}")
    
    if len(training_data) < 1000:
        print("  Small training set - consider data augmentation")
    
    if len(validation_data) < 100:
        print("  Small validation set - results may be unreliable")
    
    # Check data distribution
    train_loader = training_data.to_dataloader(train=True, batch_size=64, num_workers=0)
    val_loader = validation_data.to_dataloader(train=False, batch_size=64, num_workers=0)
    
    # Sample a batch
    train_batch = next(iter(train_loader))
    val_batch = next(iter(val_loader))
    
    train_targets = train_batch[1][0]  # y values
    val_targets = val_batch[1][0]
    
    print(f"\nTarget statistics:")
    print(f"   Train mean: {train_targets.mean():.4f}, std: {train_targets.std():.4f}")
    print(f"   Val mean: {val_targets.mean():.4f}, std: {val_targets.std():.4f}")
    
    # Check for data leakage
    train_mean = train_targets.mean().item()
    val_mean = val_targets.mean().item()
    
    if abs(train_mean - val_mean) / train_mean > 0.2:
        print("  Large difference in train/val target distributions")
    
    print("=" * 50)

def train_with_learning_rate_finder(use_all_data: bool = False):
    """Train model with automatic learning rate finding."""
    global optimization_results, df_clean_global
    
    if not optimization_results or 'best_params' not in optimization_results:
        print(" No best parameters found. Run optimization first!")
        return None
    
    print(" TRAINING WITH LEARNING RATE FINDER")
    print("=" * 50)
    
    # Create datasets
    training_data, validation_data = create_tft_datasets(
        df_clean_global, 
        use_selected_tickers=not use_all_data
    )
    
    # Create model with placeholder learning rate
    best_params = optimization_results['best_params'].copy()
    best_params['learning_rate'] = 1e-3  # Temporary
    
    model = TemporalFusionTransformer.from_dataset(
        training_data,
        loss=QuantileLoss([0.1, 0.5, 0.9]),
        optimizer="AdamW",
        **best_params
    )
    
    # Create trainer for LR finding
    trainer = pl.Trainer(
        gpus=1 if torch.cuda.is_available() else 0,
        max_epochs=1
    )
    
    # Create data loaders
    train_dataloader = training_data.to_dataloader(
        train=True, batch_size=64, num_workers=0
    )
    
    # Find optimal learning rate
    print(" Finding optimal learning rate...")
    lr_finder = trainer.tuner.lr_find(model, train_dataloader)
    
    # Get suggested learning rate
    suggested_lr = lr_finder.suggestion()
    print(f" Suggested learning rate: {suggested_lr:.2e}")
    
    # Update model with optimal learning rate
    best_params['learning_rate'] = suggested_lr
    
    # Retrain with optimal learning rate
    return train_final_model_with_custom_params(
        best_params, 
        use_all_data=use_all_data, 
        max_epochs=100
    )

def fix_training_config():
    """Fix common configuration issues that prevent model improvement."""
    global config
    
    print(" FIXING TRAINING CONFIGURATION")
    print("=" * 50)
    
    # Issue 1: Early stopping too aggressive
    original_patience = config.EARLY_STOPPING_PATIENCE
    config.EARLY_STOPPING_PATIENCE = 15
    print(f" Early stopping patience: {original_patience} → {config.EARLY_STOPPING_PATIENCE}")
    
    # Issue 2: Learning rate might be too high
    print(" Will use adaptive learning rate in enhanced training")
    
    # Issue 3: Batch size optimization
    print(" Will optimize batch size based on data size")
    
    # Issue 4: Better loss function
    print(" Will use QuantileLoss instead of MAE")
    
    # Issue 5: More training epochs
    print(" Increased default max_epochs to 100")
    
    print("\n TRAINING TIPS:")
    print("   • Use train_final_model_with_best_params() with the enhanced version")
    print("   • Monitor validation loss - should decrease gradually")
    print("   • If still no improvement, try train_with_learning_rate_finder()")
    print("   • Consider using diagnose_training_issues() to check your data")
    
    print("=" * 50)

def quick_fix_and_train():
    """Quick function to fix config and train with enhancements."""
    print("⚡ QUICK FIX AND ENHANCED TRAINING")
    print("=" * 50)
    
    # Apply fixes
    fix_training_config()
    
    # Train with enhancements
    print("\n Starting enhanced training...")
    return train_final_model_with_best_params(use_all_data=False, max_epochs=100)

def setup_cuda_deterministic():
    """Setup CUDA for deterministic training if possible."""
    import os
    
    if torch.cuda.is_available():
        try:
            # Set CUDA deterministic environment
            os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
            torch.use_deterministic_algorithms(True)
            print(" CUDA deterministic mode configured successfully")
            return True
        except Exception as e:
            print(f"  Could not enable CUDA deterministic mode: {e}")
            print("   Training will continue without deterministic guarantees")
            return False
    else:
        print("ℹ  Using CPU - deterministic mode available")
        return True

def train_non_deterministic(use_all_data: bool = False, max_epochs: int = 100):
    """Train model without deterministic constraints for maximum compatibility."""
    global optimization_results, df_clean_global
    
    if not optimization_results or 'best_params' not in optimization_results:
        print(" No best parameters found. Run optimization first!")
        return None
    
    print(" TRAINING MODEL (NON-DETERMINISTIC MODE)")
    print("=" * 70)
    
    best_params = optimization_results['best_params'].copy()
    
    # Adjust learning rate
    original_lr = best_params.get('learning_rate', 1e-3)
    final_lr = original_lr * 0.7
    best_params['learning_rate'] = final_lr
    
    # Create datasets
    training_data, validation_data = create_tft_datasets(
        df_clean_global, 
        use_selected_tickers=not use_all_data
    )
    
    # Optimize batch size
    optimal_batch_size = min(config.BATCH_SIZE, len(training_data) // 10)
    optimal_batch_size = max(optimal_batch_size, 16)
    
    # Create data loaders
    train_dataloader = training_data.to_dataloader(
        train=True, 
        batch_size=optimal_batch_size, 
        num_workers=0,
        shuffle=True
    )
    val_dataloader = validation_data.to_dataloader(
        train=False, 
        batch_size=optimal_batch_size, 
        num_workers=0
    )
    
    # Create model
    from pytorch_forecasting.metrics import QuantileLoss
    
    model = TemporalFusionTransformer.from_dataset(
        training_data,
        loss=QuantileLoss([0.1, 0.5, 0.9]),
        optimizer="AdamW",
        **best_params
    )
    
    # Create trainer WITHOUT deterministic constraints
    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=15,
            verbose=True,
            mode="min",
            min_delta=1e-6
        ),
        ModelCheckpoint(
            dirpath=config.CHECKPOINT_DIR,
            filename=f'tft-final-{datetime.now().strftime("%Y%m%d_%H%M%S")}-{{epoch:02d}}-{{val_loss:.4f}}',
            monitor='val_loss',
            mode='min',
            save_top_k=3,
            save_last=True
        )
    ]
    
    trainer = pl.Trainer(
        max_epochs=max_epochs,
        gpus=1 if torch.cuda.is_available() else 0,
        callbacks=callbacks,
        gradient_clip_val=0.1,
        enable_progress_bar=True,
        precision=32,
        # NO deterministic=True to avoid CUDA issues
    )
    
    print(f"Training started (max_epochs: {max_epochs})...")
    trainer.fit(model, train_dataloader, val_dataloader)
    
    # Get the best model
    try:
        best_model = TemporalFusionTransformer.load_from_checkpoint(
            callbacks[1].best_model_path  # ModelCheckpoint callback
        )
    except:
        best_model = model
    
    print("Training completed successfully!")
    return best_model

In [46]:
setup_cuda_deterministic()

 CUDA deterministic mode configured successfully


True

In [47]:
# Fix configuration and train with enhancements
fix_training_config()  # This fixes the early stopping and other issues

 FIXING TRAINING CONFIGURATION
 Early stopping patience: 15 → 15
 Will use adaptive learning rate in enhanced training
 Will optimize batch size based on data size
 Will use QuantileLoss instead of MAE
 Increased default max_epochs to 100

 TRAINING TIPS:
   • Use train_final_model_with_best_params() with the enhanced version
   • Monitor validation loss - should decrease gradually
   • If still no improvement, try train_with_learning_rate_finder()
   • Consider using diagnose_training_issues() to check your data


In [ ]:
model = train_non_deterministic(use_all_data=True, max_epochs=100)     # data = all,  for early_stopping = 15

 TRAINING MODEL (NON-DETERMINISTIC MODE)
 Creating datasets with features:
    Target: Close
    Group ID: Ticker
    Time features: time_idx, month_sin, month_cos
    Technical indicators: ['Volume', 'MACD', 'RSI', 'CCI', 'ADX']
    Sentiment: Sentiment_Label


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 2050') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 180   
3  | prescalers     

Training started (max_epochs: 100)...
Epoch 0: 100%|██████████| 478/478 [01:43<00:00,  4.60it/s, loss=28.1, v_num=16, train_loss_step=33.30, val_loss=65.50]

Metric val_loss improved. New best score: 65.474


Epoch 3: 100%|██████████| 478/478 [03:33<00:00,  2.24it/s, loss=28.3, v_num=16, train_loss_step=44.50, val_loss=62.20, train_loss_epoch=27.50]

Metric val_loss improved by 3.305 >= min_delta = 1e-06. New best score: 62.169


Epoch 18: 100%|██████████| 478/478 [02:03<00:00,  3.87it/s, loss=18.5, v_num=16, train_loss_step=25.40, val_loss=91.90, train_loss_epoch=17.50]

Monitored metric val_loss did not improve in the last 15 records. Best score: 62.169. Signaling Trainer to stop.


Epoch 18: 100%|██████████| 478/478 [02:03<00:00,  3.87it/s, loss=18.5, v_num=16, train_loss_step=25.40, val_loss=91.90, train_loss_epoch=17.10]
Training completed successfully!


# WASTE

In [ ]:
# Test minimal TFT creation
def test_minimal_tft():
    """Test minimal TFT model creation"""
    try:
        # Create simple test data
        data = pd.DataFrame({
            'time_idx': range(100),
            'target': np.random.randn(100),
            'group': ['A'] * 100
        })
        
        # Create minimal dataset
        dataset = TimeSeriesDataSet(
            data,
            time_idx='time_idx',
            target='target',
            group_ids=['group'],
            max_encoder_length=30,
            max_prediction_length=7,
        )
        
        # Create minimal model
        model = TemporalFusionTransformer.from_dataset(dataset)
        print(f"Model type: {type(model)}")
        print(f"Is LightningModule: {isinstance(model, pl.LightningModule)}")
        
        return True
    except Exception as e:
        print(f"Test failed: {e}")
        return False

# Run the test
test_minimal_tft()

In [ ]:
# Add this at the top after imports to check compatibility
def check_versions():
    import pytorch_forecasting
    print(f"PyTorch Forecasting version: {pytorch_forecasting.__version__}")
    
    # Create a simple test model to verify compatibility
    try:
        from pytorch_forecasting.models.temporal_fusion_transformer import TemporalFusionTransformer
        print("TFT import successful")
    except Exception as e:
        print(f"TFT import failed: {e}")

check_versions()

In [ ]:
# ==============================================================================
# STAGE 2, PART A: PREPARE INPUTS FOR PORTFOLIO OPTIMIZATION
# ==============================================================================
print("\nPreparing inputs for portfolio optimization...")

# --- 1. Calculate Expected Returns from TFT Forecasts ---
# Get model predictions
predictions = best_tft.predict(val_dataloader, mode="prediction")
# Get the tickers in the same order as the predictions
tickers = validation_data.groups.keys()
# Calculate mean expected return for each stock over the prediction horizon
expected_returns_annual = {}
current_prices = {}

# We need the last known price to calculate percentage return
for ticker in tickers:
    current_prices[ticker] = df_clean[df_clean.Ticker == ticker].sort_values('time_idx')['Close'].iloc[-1]

for i, ticker in enumerate(tickers):
    # The model predicts the future price. We convert this to a percentage return.
    mean_future_price = predictions[i].mean().item()
    current_price = current_prices[ticker]
    # Calculate expected return over the 21-day horizon
    expected_return_horizon = (mean_future_price - current_price) / current_price
    # Annualize the return (approx. 252 trading days in a year)
    expected_returns_annual[ticker] = (1 + expected_return_horizon)**(252 / max_prediction_length) - 1

returns_series = pd.Series(expected_returns_annual)

# --- 2. Calculate Covariance Matrix from Historical Data ---
# Create a pivot table of historical close prices
close_prices = df_clean.pivot(index='Date', columns='Ticker', values='Close')
# Calculate daily percentage returns
daily_returns = close_prices.pct_change().dropna()
# Calculate annualized covariance matrix
cov_matrix_annual = daily_returns.cov() * 252

print("Inputs prepared:")
print("\nAnnualized Expected Returns from TFT (%):")
print((returns_series * 100).round(2))
print("\nAnnualized Covariance Matrix (sample):")
print(cov_matrix_annual.head().round(4))

# ==============================================================================
# STAGE 2, PART B: PERFORM PORTFOLIO OPTIMIZATION
# ==============================================================================
def portfolio_performance(weights, expected_returns, cov_matrix):
    """Calculates portfolio performance stats."""
    returns = np.sum(weights * expected_returns)
    std_dev = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return returns, std_dev

def negative_sharpe_ratio(weights, expected_returns, cov_matrix, risk_free_rate=0.0):
    """Calculates the negative Sharpe ratio for the optimizer."""
    p_returns, p_std_dev = portfolio_performance(weights, expected_returns, cov_matrix)
    return -(p_returns - risk_free_rate) / p_std_dev

def portfolio_optimization(expected_returns, cov_matrix):
    """
    Performs the portfolio optimization to find the weights that maximize the Sharpe Ratio.
    """
    print("\nRunning Portfolio Optimization...")
    num_assets = len(expected_returns)
    args = (expected_returns, cov_matrix)
    # Constraints: sum of weights is 1
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    # Bounds: weights for each asset are between 0 and 1 (no short selling)
    bounds = tuple((0, 1) for _ in range(num_assets))
    # Initial guess: equal weights
    initial_guess = num_assets * [1. / num_assets]

    # Run the optimization
    result = sco.minimize(negative_sharpe_ratio, initial_guess, args=args,
                          method='SLSQP', bounds=bounds, constraints=constraints)
    return result

# Run the optimization
optimization_result = portfolio_optimization(returns_series, cov_matrix_annual)


# ==============================================================================
# STAGE 2, PART C: DISPLAY RESULTS
# ==============================================================================
def display_results(result, expected_returns):
    """Prints the final portfolio allocation and performance."""
    print("\n---------- OPTIMAL PORTFOLIO RESULTS ----------")
    # Get stats from the optimized weights
    weights = result.x
    expected_return, std_dev = portfolio_performance(weights, returns_series, cov_matrix_annual)
    sharpe_ratio = -result.fun # Un-negate the Sharpe ratio

    print(f"Max Sharpe Ratio: {sharpe_ratio:.2f}")
    print(f"Expected Annual Return: {expected_return*100:.2f}%")
    print(f"Expected Annual Volatility (Risk): {std_dev*100:.2f}%")
    print("\nOptimal Portfolio Weights:")
    results_df = pd.DataFrame({
        'Weight': (weights * 100).round(2)
    }, index=expected_returns.index)
    print(results_df[results_df.Weight > 0].sort_values('Weight', ascending=False))
    print("---------------------------------------------")

display_results(optimization_result, returns_series)